# Parser Verifier — RTL-grounded behavioral audit

Audits the behavioral claims in `data/parsed_v3_tuning18/*.json` against the authoritative
RTL in `data/RTL_data/*.vhd`.

> **The RTL is the authoritative source. The parsed JSON is a hypothesis to be checked.**

**Scope — behavioral fidelity only.** functionality · roles · relationships · relationship
targets · evidence.

**Explicitly out of scope.** No asset identification, no C/I/A/U, no security relevance, no
use of `back_test.md` or the asset labels, no security implications, no rewriting or silently
correcting parser output, no external documentation, no general NEORV32 knowledge used to
fill gaps, no assuming similarly-named signals behave alike.

The verifier is an **auditor, not a second parser**. The required reasoning is
`Parser says X -> find X in RTL -> does the RTL directly support X?`, never
`Parser says X -> what do I think X probably means?`

## Statuses

| status | meaning |
|---|---|
| `SUPPORTED` | RTL directly supports the claim |
| `PARTIALLY_SUPPORTED` | broadly correct but omits a condition, branch, mode, generate guard, limitation, or target |
| `CONTRADICTED` | RTL directly shows behavior inconsistent with the claim |
| `NOT_ESTABLISHED` | plausible, but the supplied RTL does not establish it |
| `UNVERIFIABLE` | required RTL / entity / target cannot be located or mapped confidently |

Absence of evidence is `NOT_ESTABLISHED`, **never** `CONTRADICTED`.

## Anti-drift

A verification run never reads a previous verification result. Every run independently
compares `parser output <-> RTL`. Earlier runs feed aggregate statistics only, so an early
verifier mistake cannot become pseudo-ground-truth later. This is asserted in code.

## 1. Configuration

In [1]:
import os, sys, io, re, json, time, hashlib, random, subprocess
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
import pandas as pd

HERE = Path.cwd() if Path.cwd().name != "notebooks" else Path.cwd().parent

CONFIG = {
    "parsed_dir":        str(HERE / "data" / "parsed_v3_tuning18"),
    "rtl_dir":           str(HERE / "data" / "RTL_data"),
    "results_root":      str(HERE / "verification_results"),

    # provenance recorded in the manifest
    "parser_prompt_version": "v3_tuning18",
    "parsed_data_version":   "parsed_v3_tuning18",
    "rtl_dataset_version":   "RTL_data",
    "verifier_prompt_version": "VP5",

    # Change this for every intentionally fresh replicate.
    "run_tag": "vp5_full_pass_01",

    "env_file":          str(HERE / "API.env"),
    "provider":          "openai",
    "model":             "gpt-5-mini",
    "api_key_env":       "OPENAI_API_KEY",

    # Reasoning models are driven through the Responses API with `reasoning.effort`.
    # They do NOT accept temperature or seed, so those knobs do not exist here --
    # run-to-run determinism comes from the response cache, not from decoding params.
    "reasoning_effort":  "high",  # "low" / "medium" / "high" / "max"
    # MEASURED over 99 cached calls at effort=high: output tokens p50 4384,
    # p90 10009, p95 11482, p99 13300, max 13706; reasoning is 61% of output on
    # median and up to 90%. So 12000 truncates roughly the top 5-15% of elements
    # and each truncation is a billed answer thrown away.
    #
    # max_output_tokens is a CAP, not a charge -- nothing here was ever billed at
    # the cap -- so raising it costs NOTHING for elements that fit. 16000-24000
    # would remove nearly all escalation for free.
    #
    # DO NOT raise it mid-study. It is part of the cache key, so changing it
    # orphans every cached response, and it would stop the current replicate being
    # a like-for-like repeat of run_006 (which ran at 12000). Change it when
    # opening the full pass, and record the change as a new arm in
    # VERIFIER_ABLATION_LOG.md.
    # 12000 -> 24000. Measured over 142 cached calls: p50 5851, p95 11498, p99 12361,
    # max 13706 -- a 16000 start would have truncated none of them. The cap is NOT
    # billed when unused, so raising it costs nothing for the ~99% of calls that fit,
    # and it removes the discarded-answer waste: run_009 threw away $0.0989 of $0.5334
    # (18.5%) re-asking 4 truncated calls, which is ~$6 of a $33 full pass.
    "max_output_tokens": 24000,
    # local RNG only (retry jitter). NOT an API parameter -- the Responses API
    # accepts no seed, so this does not make model output reproducible.
    "seed":              20260818,
    # reasoning tokens share the output budget, so a verbose schema can exhaust it
    # and return empty text. On `incomplete`, retry with a bigger budget.
    "escalate_on_incomplete":    True,
    "max_output_tokens_ceiling": 32000,
    # only sent for models that support it; auto-detected from the model name
    "force_reasoning":   None,      # True / False to override auto-detection
    "temperature":       0.0,       # anthropic path only; ignored by Responses API

    # per-1M-token prices for the cost report (defaults: gpt-5-mini)
    "price_in":          0.25,
    "price_cached_in":   0.025,
    "price_out":         2.00,

    "max_retries":       6,
    "backoff_base_s":    2.0,
    "backoff_cap_s":     60.0,
    "request_timeout_s": 180,
    # 4 -> 8. Measured on run_010: median call 65s, max 119s, 31.9 compute-hours for
    # 1689 elements. At 4 workers that is ~8 hours of wall clock; at 8 it is ~4. Kept
    # at 8 rather than 16 because every worker shares one rate limit and the retry
    # path treats a 429 as a transient error with exponential backoff -- pushing
    # concurrency until the limiter pushes back would trade wall clock for retries.
    "parallel_workers":  8,
    "cache_enabled":     True,

    # RTL context budget per element (characters)
    "ctx_window_lines":  6,      # +/- lines around each hit
    "ctx_max_hits":      40,     # cap hits per symbol
    "ctx_max_chars":     32000,  # hard cap on assembled RTL context

    # VP4. The claim-driven excerpt was assembled by searching the RTL for the very
    # symbols the parser claimed, so the verifier was shown evidence selected by the
    # hypothesis it was supposed to test -- and VP3 duly returned NOT_ESTABLISHED 0
    # times in 182 checks. Measured: 25 of 26 architectures fit in 32000 chars
    # (only neorv32_cpu at 23740 needed the raise), covering 100% of elements.
    # Send the whole architecture whenever it fits; fall back to the excerpt only
    # when it does not, and tell the verifier which mode it is in.
    "prefer_full_architecture": True,
}

random.seed(CONFIG["seed"])
print("parsed :", CONFIG["parsed_dir"], Path(CONFIG["parsed_dir"]).exists())
print("rtl    :", CONFIG["rtl_dir"], Path(CONFIG["rtl_dir"]).exists())

parsed : e:\jobs\ff\test\Prepoison_subset\bahavioral_patterns_of_assets\data\parsed_v3_tuning18 True
rtl    : e:\jobs\ff\test\Prepoison_subset\bahavioral_patterns_of_assets\data\RTL_data True


In [ ]:
def load_env_file(path):
    """Minimal .env reader. Tolerates `KEY = value`, quotes, comments, and a
    missing trailing newline. Values are never printed."""
    p = Path(path); loaded = []
    if not p.exists(): return loaded
    for raw in p.read_text(encoding="utf-8-sig").splitlines() or [p.read_text(encoding="utf-8-sig")]:
        line = raw.strip()
        if not line or line.startswith("#") or "=" not in line: continue
        k, v = line.split("=", 1)
        k = k.strip(); v = v.strip().strip('"').strip("'")
        if k:
            os.environ[k] = v; loaded.append(k)
    return loaded

_loaded = load_env_file(CONFIG["env_file"])
print(f"loaded from {Path(CONFIG['env_file']).name}: {_loaded or 'nothing'}")

API_KEY = os.environ.get(CONFIG["api_key_env"], "").strip()
if not API_KEY:
    raise RuntimeError(
        f"{CONFIG['api_key_env']} not found in {CONFIG['env_file']} or the environment.\n"
        f"Expected a line like  {CONFIG['api_key_env']}=sk-...  in API.env.\n"
        f"Never hard-code the key in this notebook.")
print(f"{CONFIG['api_key_env']}: present ({len(API_KEY)} chars, "
      f"ends ...{API_KEY[-4:]})")

# Reasoning models take `reasoning.effort` and reject temperature/seed.
def _is_reasoning_model(m):
    m = m.lower()
    return m.startswith(("gpt-5", "o1", "o3", "o4")) or "-reasoning" in m
USE_REASONING = (CONFIG["force_reasoning"]
                 if CONFIG["force_reasoning"] is not None
                 else _is_reasoning_model(CONFIG["model"]))
print(f"model {CONFIG['model']}: reasoning={USE_REASONING}"
      + (f", effort={CONFIG['reasoning_effort']}" if USE_REASONING else ""))

In [3]:
# ---- run directory: never overwrite a previous run ---------------------------
def next_run_dir(root):
    """Next free run_NNN, reusing a trailing EMPTY one.

    This cell mints a directory every time it executes, and it executes on every
    kernel restart and every top-to-bottom re-run -- so an interrupted session, or
    simply re-running the notebook, left behind empty run_NNN directories that look
    like real runs in a listing and push the numbering forward. (run_007 was one.)
    An empty directory carries no result, so reuse it rather than accumulating.
    """
    root = Path(root); root.mkdir(parents=True, exist_ok=True)
    existing = sorted((p for p in root.glob("run_*")
                       if p.is_dir() and p.name.split("_")[-1].isdigit()),
                      key=lambda p: int(p.name.split("_")[1]))
    for p in reversed(existing):
        if not any(p.iterdir()):
            return p                      # empty -> reuse, do not mint another
        break                             # newest is populated -> mint a fresh one
    n = 1 + max([int(p.name.split("_")[1]) for p in existing] or [0])
    d = root / f"run_{n:03d}"
    d.mkdir(parents=True, exist_ok=False)
    return d

RUN_DIR = next_run_dir(CONFIG["results_root"])
CACHE_DIR = Path(CONFIG["results_root"], "_cache"); CACHE_DIR.mkdir(parents=True, exist_ok=True)
print("run dir:", RUN_DIR)

# ---- ANTI-DRIFT GUARD (instructions section 22) ------------------------------
# No verification result from any previous run may enter this run as evidence.
_FORBIDDEN_READ = [p for p in Path(CONFIG["results_root"]).glob("run_*") if p != RUN_DIR]
def assert_no_prior_results_used(text, where):
    """Called on every payload sent to the model."""
    for p in _FORBIDDEN_READ:
        if p.name in str(text):
            raise AssertionError(f"anti-drift violation: prior run {p.name} referenced in {where}")
    for tok in ["overall_status", "functionality_check", "role_checks", "relationship_checks",
                "verification_results"]:
        if tok in str(text):
            raise AssertionError(f"anti-drift violation: verifier-output vocabulary '{tok}' in {where}")
print(f"anti-drift guard armed; {len(_FORBIDDEN_READ)} prior run(s) quarantined")

run dir: e:\jobs\ff\test\Prepoison_subset\bahavioral_patterns_of_assets\verification_results\run_011
anti-drift guard armed; 7 prior run(s) quarantined


## 2. Load parsed JSON

Schema observed in `parsed_v3_tuning18`:

```
{ "module": str, "entities": [str], "ports": [element], "signals": [element], "_issues": {...} }
element = { entity, name, type, function, functionality, role[], relationship[{type,targets[]}],
            evidence, dir?, kind? }
```

`_issues` is the parser's own self-report and is loaded as deterministic context, not as truth.

In [4]:
ELEMENT_KEYS = ["entity", "name", "type", "function", "functionality",
                "role", "relationship", "evidence", "dir", "kind"]

def load_parsed(parsed_dir):
    recs, files, malformed = [], {}, []
    for p in sorted(Path(parsed_dir).glob("*.json")):
        try:
            j = json.loads(p.read_text(encoding="utf-8"))
        except Exception as e:
            malformed.append({"file": p.name, "error": f"JSON_SYNTAX: {e}"}); continue
        files[p.stem] = {"path": str(p), "module": j.get("module", p.stem),
                         "entities": j.get("entities", []) or [],
                         "issues": j.get("_issues", {}) or {}}
        for bucket in ("ports", "signals"):
            for i, e in enumerate(j.get(bucket, []) or []):
                if not isinstance(e, dict) or "name" not in e or "entity" not in e:
                    malformed.append({"file": p.name, "bucket": bucket, "index": i,
                                      "error": "MALFORMED_RECORD: missing name/entity"})
                    continue
                r = {k: e.get(k) for k in ELEMENT_KEYS}
                r.update(source_file=p.name, module=j.get("module", p.stem),
                         bucket=bucket, index=i)
                r["role"] = r["role"] or []
                r["relationship"] = r["relationship"] or []
                recs.append(r)
    return recs, files, malformed

PARSED, FILES, MALFORMED = load_parsed(CONFIG["parsed_dir"])
for r in PARSED:
    r["element_id"] = "el_" + hashlib.sha256(
        f"{r['module']}\x00{r['entity']}\x00{r['name']}".encode()).hexdigest()[:16]

pdf = pd.DataFrame(PARSED)
print("files:", len(FILES), "| elements:", len(pdf), "| malformed:", len(MALFORMED))
print("modules:", pdf.module.nunique(), "| entities:", pdf.entity.nunique())
print("relationship types present in data:")
rt = Counter(x.get("type") for r in PARSED for x in r["relationship"])
print("  ", dict(rt.most_common()))
dupes = pdf.duplicated(subset=["module", "entity", "name"], keep=False)
print("duplicate (module,entity,name) rows:", int(dupes.sum()))
if MALFORMED: display(pd.DataFrame(MALFORMED).head())

files: 18 | elements: 1689 | malformed: 0
modules: 17 | entities: 26
relationship types present in data:
   {'SOURCES': 646, 'CARRIES': 460, 'EXPORTS': 430, 'DERIVES_FROM': 217, 'CAPTURES': 216, 'GATES': 149, 'ISOLATED': 110, 'SEQUENCES': 76, 'SLICES': 40, 'SELECTS': 39, 'AGGREGATES': 38, 'CONSTRAINS': 30, 'OVERRIDES': 11, 'REFLECTS': 2}
duplicate (module,entity,name) rows: 0


> **Relationship vocabulary discrepancy — read this.**
> The specification lists eight types: `CAPTURES DERIVES_FROM GATES SELECTS SOURCES
> CONSTRAINS CARRIES SEQUENCES`. The parsed data also uses `EXPORTS`, `ISOLATED`,
> `OVERRIDES`, `AGGREGATES`, `SLICES`, `REFLECTS`.
>
> These are **not** treated as errors — the parser may legitimately have a wider vocabulary.
> They are flagged deterministically as `type_outside_spec_vocabulary` so you can decide
> whether the parser prompt or the spec should change. The verifier is told to audit them
> literally, by their plain meaning, exactly as it does the eight specified types.

## 3. RTL index and file mapping

A `.vhd` file may hold several entities (`neorv32_bus.vhd` holds six), so mapping is
**entity -> (file, line span)**, not module -> file. An entity is only mapped when its
`entity <name> is` declaration is actually found; a filename resembling an entity name is
never sufficient. Unmapped entities are recorded `UNVERIFIABLE_FILE_MAPPING` and are not
guessed.

In [5]:
ENT_RE  = re.compile(r"(?im)^\s*entity\s+(\w+)\s+is\b")
ARCH_RE = re.compile(r"(?im)^\s*architecture\s+(\w+)\s+of\s+(\w+)\s+is\b")
END_RE  = re.compile(r"(?im)^\s*end\s+(?:entity|architecture)?\s*;?\s*$")

def build_rtl_index(rtl_dir):
    idx, files = {}, {}
    for p in sorted(Path(rtl_dir).glob("*.vhd")):
        src = p.read_text(encoding="utf-8", errors="replace")
        lines = src.split("\n")
        # comment-stripped mirror: an identifier mentioned only in a comment is NOT
        # evidence that the signal exists in the design.
        code = [l.split("--")[0] for l in lines]
        files[p.name] = {"path": str(p), "lines": lines, "code": code,
                         "sha256": hashlib.sha256(src.encode("utf-8", "replace")).hexdigest()}
        spans = []
        for m in ENT_RE.finditer(src):
            spans.append(("entity", m.group(1), src[:m.start()].count("\n")))
        for m in ARCH_RE.finditer(src):
            spans.append(("arch", m.group(2), src[:m.start()].count("\n")))
        spans.sort(key=lambda t: t[2])
        for i, (kind, name, ln) in enumerate(spans):
            end = spans[i + 1][2] if i + 1 < len(spans) else len(lines)
            e = idx.setdefault(name.lower(), {"entity": name, "file": p.name, "decl": None,
                                              "arch": None})
            if e["file"] != p.name:      # same entity name in two files -> ambiguous
                e["ambiguous"] = True
            if kind == "entity": e["decl"] = (ln, end)
            else: e["arch"] = (ln, end)
        # constants that gate generate branches (section 11)
        consts = {}
        for m in re.finditer(r"(?im)^\s*constant\s+(\w+)\s*:\s*boolean\s*:=\s*(true|false)\s*;", src):
            consts[m.group(1).lower()] = (m.group(2).lower() == "true")
        files[p.name]["bool_constants"] = consts
    return idx, files

RTL_INDEX, RTL_FILES = build_rtl_index(CONFIG["rtl_dir"])

# Union of every identifier occurring in actual RTL code (comments excluded).
# Used to decide which tokens in a prose `evidence` string are plausibly meant as
# signal references at all. Without this, prose words ("assignment", "concurrent",
# "the") get reported as missing signals and swamp the real findings.
_IDENT = re.compile(r"[A-Za-z_]\w*")
GLOBAL_RTL_IDENTS = set()
for _f in RTL_FILES.values():
    for _l in _f["code"]:
        GLOBAL_RTL_IDENTS.update(t.lower() for t in _IDENT.findall(_l))
print("rtl files:", len(RTL_FILES), "| indexed entities:", len(RTL_INDEX),
      "| distinct code identifiers:", len(GLOBAL_RTL_IDENTS))
mapped = sum(1 for e in pdf.entity.unique() if e.lower() in RTL_INDEX)
print(f"parsed entities mapped to an RTL entity declaration: {mapped}/{pdf.entity.nunique()}")
unmapped = sorted(e for e in pdf.entity.unique() if e.lower() not in RTL_INDEX)
if unmapped: print("UNVERIFIABLE_FILE_MAPPING:", unmapped)
amb = [k for k, v in RTL_INDEX.items() if v.get("ambiguous")]
if amb: print("ambiguous entity names (same name, multiple files):", amb)
print("boolean constants found (generate guards):",
      {f: c for f, d in RTL_FILES.items() if (c := d["bool_constants"])})

rtl files: 18 | indexed entities: 26 | distinct code identifiers: 1966
parsed entities mapped to an RTL entity declaration: 26/26
boolean constants found (generate guards): {'neorv32_imem.vhd': {'alt_style_c': False}}


## 4. RTL context extraction

Whole files cannot be sent (some exceed 40k chars), so per element we assemble a slice that
preserves exactly what the audit rules need:

- the **entity port clause** (so port direction/type claims are checkable);
- every line mentioning the element, its **record base** (`core_req_i.stb` -> `core_req_i`),
  and every **relationship target**, with surrounding context;
- the **enclosing construct header** for each hit — `process`, `if`, `elsif`, `case`, `when`,
  `generate` — so conditional, generate and reset context (sections 10–12) is visible rather
  than inferred;
- 1-based **line numbers** on every line, so the model can populate `rtl_location`.

This is line-oriented extraction, not a VHDL parser. When a symbol has more hits than
`ctx_max_hits`, the slice is truncated and the model is told so explicitly — a truncated
slice must yield `NOT_ESTABLISHED`, never a confident verdict.

In [6]:
ENCLOSING_RE = re.compile(
    r"(?i)\b(process|if\b|elsif\b|else\b|case\b|when\b|for\b|generate|begin|end\s+process)")

def _base(name):
    """core_req_i.stb -> core_req_i ; keeps arrays like x(3) -> x"""
    return re.split(r"[.\(]", str(name))[0].strip()

def _enclosing_headers(lines, i, look=90):
    """Walk back for the process/if/case/generate headers that contain line i."""
    out, depth_seen = [], set()
    for j in range(i, max(-1, i - look), -1):
        s = lines[j].strip()
        if not s or s.startswith("--"): continue
        m = re.match(r"(?i)^(\w+\s*:\s*)?(process|if|elsif|case|when|for)\b|.*\bgenerate\b", s)
        if m and j != i:
            key = s[:60]
            if key not in depth_seen:
                depth_seen.add(key); out.append((j + 1, lines[j].rstrip()))
        if len(out) >= 4: break
    return list(reversed(out))

def symbol_hits(lines, sym, max_hits, code=None):
    """Match against comment-stripped code so a mention inside a comment never
    counts as the signal existing. `lines` is still used for display."""
    pat = re.compile(r"(?<![\w.])" + re.escape(sym) + r"(?![\w])", re.I)
    src = code if code is not None else lines
    hits = [i for i, l in enumerate(src) if pat.search(l)]
    return hits[:max_hits], len(hits)

STD_TYPES = {"std_ulogic","std_logic","std_ulogic_vector","std_logic_vector","integer",
             "natural","boolean","bit","bit_vector","unsigned","signed","string","real"}

def _package_type_decls(entity_span_lines):
    """Record/type declarations for the non-standard types used in an entity's port
    clause, pulled from the package file(s).

    Without this the verifier sees `bus_req_i : in bus_req_t` and no definition of
    bus_req_t, so it cannot confirm any claim about a record field and raises
    UNVERIFIABLE/EVIDENCE errors that are artifacts of the excerpt, not parser
    faults. Record-typed ports are the norm in this design, so omitting these
    declarations would inject a systematic false-error mode across the whole run."""
    names = set()
    for l in entity_span_lines:
        for m in re.finditer(r":\s*(?:in|out|inout|buffer)\s+([A-Za-z_]\w*)", l, re.I):
            t = m.group(1)
            if t.lower() not in STD_TYPES: names.add(t)
    if not names: return ""
    out = []
    for pf in [f for f in RTL_FILES if "package" in f.lower()]:
        lines = RTL_FILES[pf]["lines"]
        for nm in sorted(names):
            m = re.search(r"(?im)^\s*type\s+" + re.escape(nm) + r"\s+is\b", "\n".join(lines))
            if not m: continue
            start = "\n".join(lines)[:m.start()].count("\n")
            end = start
            for j in range(start, min(start + 60, len(lines))):
                end = j
                if re.search(r"(?i)end\s+record\s*;|;\s*$", lines[j]) and j > start:
                    if re.search(r"(?i)end\s+record", lines[j]) or ";" in lines[j]: break
            out.append(f"--- type `{nm}` from {pf} ---")
            out += [f"{i+1:5d} | {lines[i].rstrip()}" for i in range(start, end + 1)]
    return "\n".join(out)

def full_architecture_context(ent, f, cfg=CONFIG):
    """The entity declaration + the whole architecture body, contiguous.

    -> (text, fits). Unabridged and unfiltered: no line was chosen because the
    parser mentioned it. That is the whole point -- it is what lets the verifier
    treat ABSENCE as evidence instead of as a gap in the excerpt.
    """
    lines = f["lines"]
    spans = []
    if ent["decl"]:
        spans.append(ent["decl"])
    if ent["arch"]:
        spans.append(ent["arch"])
    if not spans:
        return "", False
    out, prev_hi = [], None
    for lo, hi in sorted(spans):
        if prev_hi is not None and lo > prev_hi:
            out.append("        ...")
        for i in range(lo, min(hi, len(lines))):
            out.append(f"{i+1:5d} | {lines[i].rstrip()}")
        prev_hi = max(prev_hi or 0, hi)
    txt = "\n".join(out)
    return txt, len(txt) <= cfg["ctx_max_chars"]

def rtl_context_for(rec, cfg=CONFIG):
    """-> (context_text, meta). meta carries mapping status and truncation flags."""
    ent = RTL_INDEX.get(str(rec["entity"]).lower())
    meta = {"entity": rec["entity"], "rtl_file": None, "mapping": "UNVERIFIABLE_FILE_MAPPING",
            "truncated": False, "symbols": {}, "arch_span": None}
    if not ent:
        return "", meta
    f = RTL_FILES[ent["file"]]
    lines = f["lines"]
    meta.update(rtl_file=ent["file"], mapping="MAPPED", rtl_sha256=f["sha256"][:16])
    lo, hi = (ent["arch"] or ent["decl"] or (0, len(lines)))
    meta["arch_span"] = [lo + 1, hi]

    # VP4: prefer the WHOLE architecture. The excerpt below is assembled from the
    # parser's own claim, which biases the evidence toward the hypothesis under test.
    if cfg.get("prefer_full_architecture"):
        full, fits = full_architecture_context(ent, f, cfg)
        if fits:
            meta["context_mode"] = "FULL_ARCHITECTURE"
            meta["truncated"] = False
            if ent["decl"]:
                d0, d1 = ent["decl"]
                tdecl = _package_type_decls(lines[d0:d1])
                if tdecl:
                    full += ("\n\n## TYPE DECLARATIONS (from the package, for the port "
                             "types above)\n") + tdecl
                    meta["type_decls_included"] = True
            return full, meta
        meta["full_architecture_chars"] = len(full)
    meta["context_mode"] = "CLAIM_DRIVEN_EXCERPT"

    keep = set()
    # entity port clause
    if ent["decl"]:
        d0, d1 = ent["decl"]
        keep.update(range(d0, min(d1, d0 + 120)))
    syms = [str(rec["name"]), _base(rec["name"])]
    for r in rec["relationship"]:
        for t in (r.get("targets") or []):
            syms += [str(t), _base(t)]
    seen = set()
    for s in syms:
        if not s or s.lower() in seen: continue
        seen.add(s.lower())
        hits, total = symbol_hits(lines, s, cfg["ctx_max_hits"], code=f["code"])
        meta["symbols"][s] = {"hits_shown": len(hits), "hits_total": total,
                              "present": total > 0}
        if total > len(hits): meta["truncated"] = True
        for i in hits:
            keep.update(range(max(0, i - cfg["ctx_window_lines"]),
                              min(len(lines), i + cfg["ctx_window_lines"] + 1)))
            for ln, _ in _enclosing_headers(lines, i):
                keep.add(ln - 1)

    out, prev = [], None
    for i in sorted(keep):
        if prev is not None and i > prev + 1:
            out.append("        ...")
        out.append(f"{i+1:5d} | {lines[i].rstrip()}")
        prev = i
    txt = "\n".join(out)
    if len(txt) > cfg["ctx_max_chars"]:
        txt = txt[:cfg["ctx_max_chars"]] + "\n        ... [CONTEXT TRUNCATED]"
        meta["truncated"] = True
    # append record/type declarations for the entity's non-standard port types
    if ent["decl"]:
        d0, d1 = ent["decl"]
        tdecl = _package_type_decls(lines[d0:d1])
        if tdecl:
            txt += "\n\n## TYPE DECLARATIONS (from the package, for the port types above)\n" + tdecl
            meta["type_decls_included"] = True
    return txt, meta

_probe = PARSED[0]
_c, _m = rtl_context_for(_probe)
print("probe:", _probe["module"], "/", _probe["entity"], "/", _probe["name"])
print("meta :", json.dumps({k: v for k, v in _m.items() if k != "symbols"}, default=str))
print("symbols:", json.dumps(_m["symbols"])[:300])
print("context chars:", len(_c))
print("\n".join(_c.split("\n")[:14]))

probe: neorv32_boot_rom / neorv32_boot_rom / clk_i
meta : {"entity": "neorv32_boot_rom", "rtl_file": "neorv32_boot_rom.vhd", "mapping": "MAPPED", "truncated": false, "arch_span": [27, 70], "rtl_sha256": "0c62b5cd914953b1", "context_mode": "FULL_ARCHITECTURE", "type_decls_included": true}
symbols: {}
context chars: 3738
   18 | 
   19 | entity neorv32_boot_rom is
   20 |   port (
   21 |     clk_i     : in  std_ulogic; -- global clock line
   22 |     rstn_i    : in  std_ulogic; -- async reset, low-active
   23 |     bus_req_i : in  bus_req_t;  -- bus request
   24 |     bus_rsp_o : out bus_rsp_t   -- bus response
   25 |   );
   26 | end neorv32_boot_rom;
   27 | 
   28 | architecture neorv32_boot_rom_rtl of neorv32_boot_rom is
   29 | 
   30 |   -- determine physical ROM size in WORDS (expand to next power of two) --
   31 |   constant boot_rom_size_index_c : natural := index_size_f((bootloader_init_size_c/4)); -- address with (words)


## 5. Deterministic checks (run before the LLM)

The static layer must not decide anything semantic. If it labelled reset-guarded assignments
or unresolved citations as *errors*, it would be telling the verifier what the parser got
wrong before any RTL comparison happened — which defeats the point of an independent audit.
So its output is split three ways, and only the first is a judgement:

| bucket | contents | weight given to the verifier |
|---|---|---|
| **confirmed problems** | duplicate record key, malformed record, entity absent from the RTL set | mechanical, indisputable |
| **facts** | assignment style (`combinational`/`clocked`/`both`/`none`), reset-guarded assignment, generate guards and their constant values, symbol-resolution tier | reliable measurements, no verdict |
| **review hints** | reset-branch assignment under a `SEQUENCES` claim, generate-guarded target, unresolved citation, non-spec relationship type | **questions only** — carry no presumption of error |

Every hint ships with the question the verifier must answer, and the prompt states that a hint
is not evidence and must never be cited as justification.

### Symbol resolution is graded, not binary

`found / not found` was wrong: a cited `bus_req_i.priv` may live in a package type
declaration, reach the entity through a port map, or belong to another entity entirely.
`resolve_symbol` returns the strongest tier it can establish:

`DIRECT_SYMBOL_MATCH` → `RECORD_BASE_MATCH` → `RECORD_FIELD_MATCH` → `PORT_MAP_MATCH` →
`PACKAGE_TYPE_MATCH` → `CROSS_ENTITY_MATCH` → `NOT_ESTABLISHED_BY_STATIC_CHECK`

The last tier means *the static matcher failed*, not *the symbol is absent*. It is passed as a
hint, never as an evidence error.

In [7]:
SPEC_TYPES = {"CAPTURES","DERIVES_FROM","GATES","SELECTS","SOURCES","CONSTRAINS",
              "CARRIES","SEQUENCES"}
IDENT_RE = re.compile(r"[A-Za-z_]\w*(?:\.\w+)*")
VHDL_KW = set("""and or not xor nand nor if then else elsif end when case is of process begin
signal port map in out inout std_ulogic std_logic vector downto to others rising_edge
falling_edge if_else generate constant variable type record integer boolean natural true
false null return function procedure architecture entity use library work all""".split())

# English prose that appears in the parser's evidence sentences. These are never
# signal references, and some of them do occur inside VHDL string literals, so
# the global-identifier test alone does not exclude them.
PROSE_STOPWORDS = set("""the this that these those into onto from with without for and but
not are was were its it has have had been being when where which while whether then than
also only both each every any all via per use used uses using set sets same other another
such more most less least internal external direct directly indirect indirectly
respectively otherwise however therefore here there they them their one two three first
second next last new old full empty high low active inactive value values bit bits byte
e.g i.e etc vs eg ie
bytes word words line lines cycle cycles time times mode modes state states side sides
above below across between during before after within under over out off down up left
right whole part parts field fields list lists item items note notes case cases""".split())

DUP_KEYS = Counter((r["module"], r["entity"], r["name"]) for r in PARSED)

def _assignment_style(lines, sym, span):
    lo, hi = span
    clocked = comb = False
    pat = re.compile(r"(?<![\w.])" + re.escape(_base(sym)) + r"\b[^<]*<=", re.I)
    for i in range(lo, min(hi, len(lines))):
        if not pat.search(lines[i]): continue
        ctx = "\n".join(lines[max(lo, i - 40):i + 1]).lower()
        tail = ctx.rsplit("process", 1)
        if "rising_edge" in ctx or "falling_edge" in ctx:
            if len(tail) > 1 and ("rising_edge" in tail[1] or "falling_edge" in tail[1]):
                clocked = True
            else: comb = True
        else: comb = True
    return "both" if (clocked and comb) else ("clocked" if clocked else
           ("combinational" if comb else "none"))

def resolve_symbol(sym, own_file):
    """Graded static resolution of a cited identifier.

    Returns the STRONGEST match tier found, never a verdict. `NOT_ESTABLISHED_BY
    _STATIC_CHECK` means the static checker could not place the symbol -- it does
    NOT mean the parser is wrong; the verifier decides that from the RTL.
    """
    base = _base(sym); tail = sym.split(".")[-1] if "." in sym else None
    own = RTL_FILES[own_file]["code"]; blob = "\n".join(own)
    W = lambda s, txt: re.search(r"(?<![\w.])" + re.escape(s) + r"(?![\w])", txt, re.I)

    if W(sym, blob):                          return "DIRECT_SYMBOL_MATCH", own_file
    if W(base, blob):                         return "RECORD_BASE_MATCH", own_file
    if tail and re.search(r"\." + re.escape(tail) + r"(?![\w])", blob, re.I):
        return "RECORD_FIELD_MATCH", own_file
    # instance port maps: `.foo => bar` or `foo => bar`
    if re.search(r"(?<![\w.])" + re.escape(base) + r"\s*=>", blob, re.I):
        return "PORT_MAP_MATCH", own_file
    # package / shared type declarations
    for pf in [f for f in RTL_FILES if "package" in f.lower()]:
        pb = "\n".join(RTL_FILES[pf]["code"])
        if W(sym, pb) or W(base, pb) or (tail and re.search(r"\." + re.escape(tail) + r"(?![\w])", pb, re.I)) \
           or (tail and W(tail, pb)):
            return "PACKAGE_TYPE_MATCH", pf
    # any other entity's source
    for of in RTL_FILES:
        if of == own_file: continue
        ob = "\n".join(RTL_FILES[of]["code"])
        if W(sym, ob) or W(base, ob):
            return "CROSS_ENTITY_MATCH", of
    return "NOT_ESTABLISHED_BY_STATIC_CHECK", None


def deterministic_checks(rec, ctx_meta):
    """-> (problems, facts, hints)

    problems = genuinely mechanical defects. No semantic judgement is possible to
               dispute them (malformed record, duplicate key, entity not in RTL).
    facts    = measurements. Assignment style, reset-guarded assignment, generate
               guards, symbol resolution tier. No verdict attached.
    hints    = "look at this" pointers for the verifier. Explicitly NOT claims
               that the parser is wrong; each carries the question to answer.

    Nothing semantic is ever placed in `problems`. Whether a reset-guarded
    SEQUENCES claim, an inactive-generate reference, or an unresolvable citation
    is actually an error is a semantic judgement, and that belongs to the verifier
    reading the RTL -- not to this layer.
    """
    problems, hints = [], []
    if DUP_KEYS[(rec["module"], rec["entity"], rec["name"])] > 1:
        problems.append({"check": "duplicate_element", "detail":
                         "more than one parsed record shares (module, entity, name)"})
    if ctx_meta["mapping"] != "MAPPED":
        problems.append({"check": "UNVERIFIABLE_FILE_MAPPING",
                         "detail": f"no `entity {rec['entity']} is` declaration found in RTL_data"})
        return problems, {}, hints
    ent = RTL_INDEX[str(rec["entity"]).lower()]
    lines = RTL_FILES[ent["file"]]["lines"]
    consts = RTL_FILES[ent["file"]]["bool_constants"]
    span = ent["arch"] or ent["decl"] or (0, len(lines))

    tier, where = resolve_symbol(str(rec["name"]), ent["file"])
    if tier == "NOT_ESTABLISHED_BY_STATIC_CHECK":
        hints.append({"hint": "element_name_unresolved_by_static_check",
                      "detail": f"static checks could not place `{rec['name']}` in {ent['file']}",
                      "question": "Does the element appear in the RTL excerpt under another "
                                  "form (record field, aggregate, port map)? If genuinely absent, "
                                  "that bears on every claim about it."})

    facts = {"rtl_file": ent["file"], "arch_span": ctx_meta["arch_span"],
             "element_symbol_resolution": {"tier": tier, "found_in": where},
             "assignment_style_of_element": _assignment_style(lines, rec["name"], span),
             "targets": {}}

    for r in rec["relationship"]:
        t = str(r.get("type", ""))
        # Relationship vocabulary is a RUN-LEVEL property of the parser, not a
        # per-element observation. Emitting it per element produced 631 identical
        # hints and risked training the verifier to read "non-spec" as "wrong".
        # It is reported once, in the manifest; the prompt tells the verifier to
        # audit any type literally regardless of vocabulary.
        for tgt in (r.get("targets") or []):
            ttier, twhere = resolve_symbol(str(tgt), ent["file"])
            present = ttier != "NOT_ESTABLISHED_BY_STATIC_CHECK"
            info = {"symbol_resolution": {"tier": ttier, "found_in": twhere},
                    "assignment_style": None, "assigned_under_reset": False,
                    "in_generate_branch": None}
            if present and twhere == ent["file"]:
                info["assignment_style"] = _assignment_style(lines, tgt, span)
                pat = re.compile(r"(?<![\w.])" + re.escape(_base(tgt)) + r"\b[^<]*<=", re.I)
                for i in range(span[0], min(span[1], len(lines))):
                    if not pat.search(lines[i]): continue
                    back = "\n".join(lines[max(span[0], i - 25):i]).lower()
                    if re.search(r"if\s+.*rstn?\w*\s*=\s*'0'|if\s+.*\breset\b", back):
                        info["assigned_under_reset"] = True
                    g = re.findall(r"if\s+(\w+)\s+generate", back)
                    if g:
                        info["in_generate_branch"] = {gg: consts.get(gg.lower()) for gg in g}
            facts["targets"][str(tgt)] = info

            if not present:
                hints.append({"hint": "target_unresolved_by_static_check",
                              "detail": f"relationship {t} target `{tgt}` was not placed by any "
                                        f"static tier (direct / record base / record field / "
                                        f"port map / package / cross-entity)",
                              "question": "Locate the target in the RTL excerpt yourself. If it "
                                          "truly is not referenced, judge the relationship "
                                          "accordingly; if it is present in a form the static "
                                          "check missed, ignore this hint."})
            # Scope: the section-12 concern is a reset dependency being described as
            # ordinary sequencing, i.e. `rstn_i SEQUENCES ctrl`. Firing on every
            # relationship whose target happens to touch a reset branch produced 1570
            # hints (~1 per relationship) and drowned the signal. Fire only when the
            # source element is reset-like, or the claimed type is one that asserts
            # clock-driven state progression.
            _src_is_reset = bool(re.search(r"(?i)(^|_)(rstn?|reset)(_|$|\d)", str(rec["name"])))
            if info["assigned_under_reset"] and (_src_is_reset or
                                                 t.upper() in {"SEQUENCES", "CAPTURES"}):
                hints.append({"hint": "target_assigned_in_reset_branch",
                              "detail": f"`{tgt}` has at least one assignment inside a "
                                        f"reset-guarded branch; parser claims `{t}`",
                              "question": f"Does `{t}` accurately describe the dependency between "
                                          f"`{rec['name']}` and `{tgt}`? A reset that DRIVES a "
                                          f"register is not the same as one that SEQUENCES it. "
                                          f"Both readings are possible -- decide from the RTL."})
            if info["in_generate_branch"]:
                gv = info["in_generate_branch"]
                inactive = any(v is False for v in gv.values())
                hints.append({"hint": "target_in_generate_branch",
                              "detail": f"`{tgt}` is assigned under generate guard(s) {gv}"
                                        + (" — guard constant is FALSE, so that branch is not "
                                           "active in the default configuration" if inactive else
                                           " (guard value is not a literal boolean constant here)"),
                              "question": "A parser may correctly describe an alternative "
                                          "implementation IF it presents it as conditional. This "
                                          "is an error only if inactive behaviour is presented as "
                                          "the active/default behaviour. Judge the phrasing."})

    # ---- evidence citations -------------------------------------------------
    # This check exists to catch the parser citing a signal that does not exist.
    # It must be conservative: a false "missing signal" is fed to the verifier as
    # reliable evidence and would corrupt the audit. Three filters, all needed:
    #   1. prose stopwords are never signal references
    #   2. the token must look like an identifier the parser meant to cite
    #   3. presence is matched LOOSELY -- a cited `addr` is satisfied by
    #      `bus_req_i.addr`, and a cited `dev_00` by `dev_00_req_o`. Only a token
    #      absent even under loose matching is reported.
    ev = str(rec.get("evidence") or "")
    code_blob = "\n".join(RTL_FILES[ent["file"]]["code"])
    ev_res = {}
    for ident in set(IDENT_RE.findall(ev)):
        low = ident.lower().strip("_")
        if not low or low in VHDL_KW or low in PROSE_STOPWORDS or len(low) < 3: continue
        plausible = ("_" in ident) or ("." in ident) or (low in GLOBAL_RTL_IDENTS)
        if not plausible: continue
        if re.search(r"(?<![A-Za-z0-9])" + re.escape(low), code_blob, re.I):
            continue                                   # resolved in own file, nothing to say
        etier, ewhere = resolve_symbol(ident, ent["file"])
        ev_res[ident] = {"tier": etier, "found_in": ewhere}
        if etier == "NOT_ESTABLISHED_BY_STATIC_CHECK":
            hints.append({"hint": "evidence_citation_unresolved_by_static_check",
                          "detail": f"evidence cites `{ident}`; no static tier placed it "
                                    f"(direct / record base / record field / port map / "
                                    f"package / cross-entity)",
                          "question": "Check the RTL excerpt directly. The citation may still be "
                                      "a valid paraphrase, or reference a form the static check "
                                      "missed. Only call it an evidence error if the RTL shows "
                                      "the cited construct is absent."})
        else:
            hints.append({"hint": "evidence_citation_resolved_elsewhere",
                          "detail": f"evidence cites `{ident}`, not found in {ent['file']} but "
                                    f"resolved as {etier} in {ewhere}",
                          "question": "Is the parser citing a construct that belongs to a "
                                      "different entity or to a package type declaration? That "
                                      "may be legitimate or may be a cross-entity error."})
    if ev_res: facts["evidence_symbol_resolution"] = ev_res
    return problems, facts, hints

_f, _facts, _hints = deterministic_checks(_probe, _m)
print("problems (mechanical only):", json.dumps(_f, indent=1)[:300])
print("review hints             :", json.dumps(_hints, indent=1)[:700])
print("facts                    :", json.dumps(_facts, indent=1)[:600])

problems (mechanical only): []
review hints             : [
 {
  "hint": "target_assigned_in_reset_branch",
  "detail": "`rdata` has at least one assignment inside a reset-guarded branch; parser claims `SEQUENCES`",
  "question": "Does `SEQUENCES` accurately describe the dependency between `clk_i` and `rdata`? A reset that DRIVES a register is not the same as one that SEQUENCES it. Both readings are possible -- decide from the RTL."
 },
 {
  "hint": "target_assigned_in_reset_branch",
  "detail": "`rden` has at least one assignment inside a reset-guarded branch; parser claims `SEQUENCES`",
  "question": "Does `SEQUENCES` accurately describe the dependency between `clk_i` and `rden`? A reset that DRIVES a register is not the same as one that SEQUENCE
facts                    : {
 "rtl_file": "neorv32_boot_rom.vhd",
 "arch_span": [
  27,
  70
 ],
 "element_symbol_resolution": {
  "tier": "DIRECT_SYMBOL_MATCH",
  "found_in": "neorv32_boot_rom.vhd"
 },
 "assignment_style_of_element": "none"

In [8]:
# ---- run deterministic layer over everything (no API cost) -------------------
DET = {}
for r in PARSED:
    _, m = rtl_context_for(r)
    fnd, facts, hts = deterministic_checks(r, m)
    DET[r["element_id"]] = {"findings": fnd, "facts": facts, "hints": hts, "ctx_meta":
                            {k: v for k, v in m.items() if k != "symbols"}}
cnt = Counter(f["check"] for d in DET.values() for f in d["findings"])
ncnt = Counter(n["hint"] for d in DET.values() for n in d["hints"])
tiers = Counter()
for d in DET.values():
    er = (d["facts"] or {}).get("element_symbol_resolution") or {}
    if er.get("tier"): tiers[er["tier"]] += 1
    for t in ((d["facts"] or {}).get("targets") or {}).values():
        tiers[(t.get("symbol_resolution") or {}).get("tier", "?")] += 1
print("CONFIRMED DETERMINISTIC PROBLEMS (mechanical, not semantic):")
for k, v in cnt.most_common():
    print("   %-34s %5d" % (k, v))
if not cnt: print("   (none)")
print()
print("REVIEW HINTS (questions for the verifier, NOT assertions of error):")
for k, v in ncnt.most_common():
    print("   %-44s %5d" % (k, v))
print()
print("DETERMINISTIC FACT: symbol resolution tier (elements + relationship targets):")
for k, v in tiers.most_common():
    print("   %-44s %5d" % (k, v))
det_df = pd.DataFrame(
    [{"item": k, "count": v, "class": "confirmed_problem"} for k, v in cnt.most_common()]
    + [{"item": k, "count": v, "class": "review_hint"} for k, v in ncnt.most_common()]
    + [{"item": k, "count": v, "class": "fact_resolution_tier"} for k, v in tiers.most_common()])
det_df

CONFIRMED DETERMINISTIC PROBLEMS (mechanical, not semantic):
   (none)

REVIEW HINTS (questions for the verifier, NOT assertions of error):
   target_assigned_in_reset_branch                512
   target_in_generate_branch                      208
   evidence_citation_resolved_elsewhere           129
   evidence_citation_unresolved_by_static_check     7

DETERMINISTIC FACT: symbol resolution tier (elements + relationship targets):
   DIRECT_SYMBOL_MATCH                           3540
   RECORD_BASE_MATCH                             1078


,item,count,class
0,target_assigned_in_reset_branch,512,review_hint
1,target_in_generate_branch,208,review_hint
2,evidence_citation_resolved_elsewhere,129,review_hint
3,evidence_citation_unresolved_by_static_check,7,review_hint
4,DIRECT_SYMBOL_MATCH,3540,fact_resolution_tier
5,RECORD_BASE_MATCH,1078,fact_resolution_tier


## 5b. Did the parser MISS anything? (deterministic, no API cost)

Every other check in this notebook audits claims the parser **made**. None of them can see a
signal the parser never mentioned — the verifier is only ever shown elements that exist in the
parsed JSON, so an omission is invisible to it by construction.

This closes that gap in the other direction, mechanically:

| | |
|---|---|
| **omitted** | declared in the RTL, no parsed element for it — the parser missed it |
| **fabricated** | a parsed element with no matching RTL declaration — the parser invented it |

Scope is deliberately narrow: **ports** from each `entity X is` clause and **signals** from its
architecture body. Generics, constants, types, components, variables and aliases are excluded —
the parser is not asked to describe them, so counting them would manufacture false omissions.
Record fields (`bus_req_i.addr`) are folded to their base port (`bus_req_i`).

Two extraction traps this code handles, both of which produced wrong answers first:

- Declarations are **not one per line**. `neorv32_bus.vhd` writes
  `dev_00_req_o : out bus_req_t; dev_00_rsp_i : in bus_rsp_t;` — a line-anchored regex drops
  every second port and reported 32 phantom fabrications.
- The **first** port sits immediately after `port (` with no preceding `;`, so a `(?:^|;)`
  delimiter silently drops one port from every entity and reported `clk_i` as fabricated 26
  times.

Requiring a mode keyword (`in`/`out`/`inout`/`buffer`) is what separates ports from generics,
so the generic clause needs no special case.

In [9]:
_DECL_NAMES = r"([A-Za-z_]\w*(?:\s*,\s*[A-Za-z_]\w*)*)"
# Delimiter is ';' OR the '(' opening the port clause -- the first port has no
# preceding semicolon. re.M so '^' also matches line starts.
PORT_DECL_RE = re.compile(r"(?:^|[;(])\s*" + _DECL_NAMES + r"\s*:\s*(?:in|out|inout|buffer)\b",
                          re.I | re.M)
SIG_DECL_RE  = re.compile(r"(?:^|[;\n])\s*signal\s+" + _DECL_NAMES + r"\s*:", re.I | re.M)

def rtl_declared_symbols(entity):
    """-> {"ports": set, "signals": set} declared for one entity, or None if unmapped.

    Matched against the COMMENT-STRIPPED mirror, so an identifier that appears only in
    a comment is never counted as a declaration."""
    info = RTL_INDEX.get(str(entity).lower())
    if not info:
        return None
    code = RTL_FILES[info["file"]]["code"]
    out = {"ports": set(), "signals": set(), "file": info["file"]}
    if info["decl"]:
        d0, d1 = info["decl"]
        for m in PORT_DECL_RE.finditer("\n".join(code[d0:min(d1, len(code))])):
            for n in m.group(1).split(","):
                out["ports"].add(n.strip().lower())
    if info["arch"]:
        a0, a1 = info["arch"]
        for m in SIG_DECL_RE.finditer("\n".join(code[a0:min(a1, len(code))])):
            for n in m.group(1).split(","):
                out["signals"].add(n.strip().lower())
    return out

def omission_report(parsed=None):
    """Parser coverage of the RTL, per entity. No API calls."""
    recs = PARSED if parsed is None else parsed
    by_ent = {}
    for r in recs:
        by_ent.setdefault((r["module"], r["entity"]), set()).add(
            str(r["name"]).split(".")[0].lower())
    rows = []
    for (mod, ent), got in sorted(by_ent.items()):
        decl = rtl_declared_symbols(ent)
        if decl is None:
            rows.append({"module": mod, "entity": ent, "status": "NO_RTL_ENTITY",
                         "rtl_declared": 0, "parsed_bases": len(got),
                         "omitted": "", "fabricated": ""})
            continue
        rtl = decl["ports"] | decl["signals"]
        om, fab = sorted(rtl - got), sorted(got - rtl)
        rows.append({"module": mod, "entity": ent,
                     "status": "OK" if not om and not fab else "MISMATCH",
                     "rtl_declared": len(rtl), "parsed_bases": len(got),
                     "omitted": ", ".join(om), "fabricated": ", ".join(fab)})
    return pd.DataFrame(rows)

OMISSIONS = omission_report()
_om = sum(len([x for x in s.split(",") if x.strip()]) for s in OMISSIONS.omitted)
_fab = sum(len([x for x in s.split(",") if x.strip()]) for s in OMISSIONS.fabricated)
print("PARSER COVERAGE OF THE RTL (deterministic, no API cost)")
print(f"  entities checked         : {len(OMISSIONS)}")
print(f"  RTL symbols declared     : {int(OMISSIONS.rtl_declared.sum())}")
print(f"  OMITTED  (in RTL, not parsed) : {_om}")
print(f"  FABRICATED (parsed, not in RTL): {_fab}")
if _om or _fab:
    print()
    print("  entities with a mismatch:")
    for _, r in OMISSIONS[OMISSIONS.status != "OK"].iterrows():
        print(f"    {r.entity:26s} omitted[{r.omitted[:60]}] fabricated[{r.fabricated[:60]}]")
else:
    print("  every declared port and signal has a parsed element, and vice versa.")
print()
print("  NOTE: this measures COVERAGE, not correctness. An element can be present and")
print("  still be described wrongly -- that is what the LLM verifier is for.")
OMISSIONS

PARSER COVERAGE OF THE RTL (deterministic, no API cost)
  entities checked         : 26
  RTL symbols declared     : 444
  OMITTED  (in RTL, not parsed) : 0
  FABRICATED (parsed, not in RTL): 0
  every declared port and signal has a parsed element, and vice versa.

  NOTE: this measures COVERAGE, not correctness. An element can be present and
  still be described wrongly -- that is what the LLM verifier is for.


,module,entity,status,rtl_declared,parsed_bases,omitted,fabricated
0,neorv32_boot_rom,neorv32_boot_rom,OK,6,6,,
1,neorv32_bus,neorv32_bus_amo_rmw,OK,13,13,,
2,neorv32_bus,neorv32_bus_amo_rvs,OK,9,9,,
3,neorv32_bus,neorv32_bus_gateway,OK,17,17,,
4,neorv32_bus,neorv32_bus_io_switch,OK,72,72,,
5,neorv32_bus,neorv32_bus_reg,OK,6,6,,
6,neorv32_bus,neorv32_bus_switch,OK,17,17,,
7,neorv32_cache,neorv32_cache,OK,10,10,,
8,neorv32_cache,neorv32_cache_memory,OK,22,22,,
9,neorv32_cpu,neorv32_cpu,OK,35,35,,


## 6. Verifier prompt (versioned)

In [10]:
VERIFIER_PROMPT_VERSION = CONFIG["verifier_prompt_version"]
VERIFIER_SYSTEM = r"""
You are an RTL auditor. You verify whether behavioral claims made by an automated parser are
actually supported by the VHDL source. You are NOT a parser and you do not produce new
behavioral descriptions of your own.

# Source hierarchy — strict
1. The supplied RTL excerpt is the authoritative source.
2. The parser claims are a hypothesis to be checked.
3. Nothing else may be used as factual evidence.

Never use external documentation. Never use general knowledge of NEORV32 or of any CPU to
fill a gap. Never assume a signal behaves a certain way because its name resembles another
signal's name. If the supplied RTL does not establish a claim, say so — do not invent an
explanation.

# Absolutely out of scope
Do not identify security assets. Do not assign C/I/A/U. Do not decide whether anything is
security-relevant, exploitable, an attack path, an information leak, or affects integrity.
Do not rewrite, improve, or silently correct the parser output. Your only sentence form is:
"the parser claims X; the RTL supports / partially supports / contradicts / does not
establish X."

# Statuses — use exactly these
- SUPPORTED — the RTL directly supports the claim.
- PARTIALLY_SUPPORTED — broadly correct but omits an important condition, branch, mode,
  generate guard, limitation, or target.
- CONTRADICTED — the RTL directly shows behavior inconsistent with the claim.
- NOT_ESTABLISHED — plausible, but the supplied RTL does not provide sufficient evidence.
- UNVERIFIABLE — the required RTL, entity, or target is not present in what you were given.

Absence of evidence is NOT_ESTABLISHED or UNVERIFIABLE. It is NEVER CONTRADICTED.
If the context is marked truncated and the missing part would decide the claim, use
NOT_ESTABLISHED rather than guessing.

# What the RTL you are given actually is — read the CONTEXT MODE

The payload states a CONTEXT MODE. It changes what absence means, so check it first.

FULL_ARCHITECTURE — you have the COMPLETE entity declaration and the COMPLETE
architecture body, contiguous and unabridged. Nothing was filtered and no line was
chosen because the parser mentioned it. In this mode ABSENCE IS EVIDENCE. If a claimed
signal, assignment, condition, branch or dependency does not appear anywhere in what you
were given, it is not in this entity: answer NOT_ESTABLISHED, and answer CONTRADICTED
when the RTL positively shows behaviour inconsistent with the claim. Do not soften a
verdict on the theory that the deciding line might be somewhere you cannot see. It is not.

CLAIM_DRIVEN_EXCERPT — the architecture was too large to send whole, so you have a slice
assembled by SEARCHING FOR THE SYMBOLS THE PARSER NAMED. The lines are there BECAUSE the
parser asserted them, which biases the evidence toward the claim you are testing. Absence
from this slice is NOT evidence of absence. Use NOT_ESTABLISHED for "not confirmable from
what I was given", and reserve CONTRADICTED for RTL you can actually quote.

NOT_ESTABLISHED is a normal, expected verdict, not a failure. A claim the RTL simply does
not support is exactly what this audit exists to find.

CONTRADICTED requires POSITIVE RTL evidence that the claim is false — RTL you can point at
and quote. A failed static check, an unresolved symbol, or a review hint is never sufficient
grounds for CONTRADICTED. If a static check could not place something but the RTL neither
shows nor refutes it, that is NOT_ESTABLISHED.

# Review hints are questions, not findings
The payload may contain REVIEW HINTS. A hint means an automated check noticed something worth
your attention — a reset-guarded assignment, a generate guard, an unresolved citation. A hint
is NOT evidence that the parser is wrong, and it carries no presumption either way. Answer the
hint's question from the RTL. If the RTL supports the parser's claim, mark it SUPPORTED even
though a hint was raised. Never cite a hint as your justification; cite RTL.

# Judge semantics, not wording, and not abstraction level
Do not mark a claim wrong merely because the parser's wording differs from the RTL, or because
it describes behaviour at a higher level of abstraction than the statements themselves. Judge
semantic equivalence. Equally, do not accept a claim merely because it sounds plausible.

Both of these are acceptable paraphrases, not errors:
  RTL:    cache_o.we <= host_req_i.ben;
  parser: "host_req_i.ben provides the byte-enable mask used for cache writes."
  RTL:    ctrl_nxt.buf_req <= ctrl.buf_req or host_req_i.stb;
  parser: "host_req_i.stb records a pending request."
Distinguish *different wording* from *different behaviour*. Only the second is an error.

# You may trace short local dependency chains — under strict limits
You are not restricted to the single line the parser happened to cite. To check a claim you
may and should read the surrounding expression, the enclosing process, and the local
assignments feeding the signals involved — anything visible in the supplied excerpt. Do not
conclude NOT_ESTABLISHED merely because the parser's cited line was incomplete, when the
surrounding RTL you were given does establish the behaviour.

Limits, all binding:
- Trace at most 3 hops, and ONLY to establish the parser's stated claim. Tracing is not
  exploration; you are not mapping the design.
- Every hop must be a concrete RTL statement you can quote and cite by line number.
- Stop at the final traced RTL relationship. Do NOT infer consequences beyond it, and do not
  extrapolate what the design "therefore" does.
- Nothing outside the supplied excerpt may enter the chain.
- If a chain needs a hop you cannot see, stop and answer NOT_ESTABLISHED.

When you trace, record it in `rtl_trace` as an ordered list of DEPENDENCY HOPS -- one entry
per signal-to-signal step in the chain, each with its line reference. `rtl_trace` is not a
bibliography: if a single step is supported by several places in the RTL, keep it as ONE entry
and put the extra locations in that entry's `rtl_location` (e.g. "L110-L116; L243-L246").
A chain of 3 hops has 3 entries. An untraceable claim is NOT_ESTABLISHED, never CONTRADICTED.

# Classify the KIND of evidence behind every status
Each check carries `evidence_type`, exactly one of:
- `DIRECT_RTL`     — RTL statements demonstrate the claim outright.
- `TRACED_RTL`     — you had to FOLLOW A DEPENDENCY from one signal to another to establish
                     the claim; `rtl_trace` lists those hops.
- `INTERPRETATION` — the RTL is clear, but deciding whether it matches the parser's *wording*
                     required a semantic judgement on your part.

Choose between DIRECT_RTL and TRACED_RTL by **whether you traversed a dependency chain**, NOT
by how many lines you cited. Citing several locations for one fact is still `DIRECT_RTL` —
put those locations in `rtl_location` (e.g. "L101; L114-L126; L170-L176") and leave
`rtl_trace` as []. Only signal-to-signal traversal makes it `TRACED_RTL`.

Hard rule: `TRACED_RTL` REQUIRES at least one entry in `rtl_trace`. If you cannot name the
hops you followed, then you did not trace anything — use `DIRECT_RTL` instead. A `TRACED_RTL`
with an empty `rtl_trace` is an invalid response and will be rejected.

Only DIRECT_RTL and TRACED_RTL are factual evidence. INTERPRETATION marks your own judgement
call, and you must use it honestly — when a status turns on how the parser phrased something
rather than on what the RTL does, say INTERPRETATION. This distinction is what later separates
"the parser was inaccurate" from "the verifier made an interpretive call", so do not inflate
a judgement into DIRECT_RTL.

The operational test, because this label has been getting rubber-stamped: DIRECT_RTL means a
careful reader given the same lines would reach the same verdict, every time. If your verdict
turns on a matter of degree — whether an omitted condition is important enough to make a claim
PARTIALLY_SUPPORTED instead of SUPPORTED, whether the parser's wording is "close enough" —
then it would NOT reproduce reliably, and it is INTERPRETATION. Label it so and fill in the
`interpretation` field. Choosing between SUPPORTED and PARTIALLY_SUPPORTED is very often an
interpretive call; say so when it is. Marking every check DIRECT_RTL destroys the only signal
that separates fact from opinion in this audit.

# What is under audit, and what is not
Audit exactly five things: functionality, each role, each relationship (type AND target),
and the evidence. The supplied `declared_type`, `declared_dir` and `declared_kind` are
CONTEXT to help you read the RTL -- they are NOT claims under audit. Do not raise errors
about them and do not mark a check UNVERIFIABLE because a type declaration is absent; judge
the behavioural claim instead.

# Claim-by-claim
Never judge the element with one global "correct"/"incorrect". Decompose:
- functionality — one status
- each role — its own status, independently
- each relationship (type AND target together) — its own status
- evidence — one status

## functionality
Check what the element actually carries; what assigns it; what reads it; whether it is
combinational or registered; whether it is captured or continuously propagated; what logic it
affects and what affects it; whether behavior changes under reset, under conditionals, or
under generate statements; and whether the description wrongly merges alternative
implementations into one unconditional description.

## roles
Verify each role independently. Ask whether the RTL supports it, whether it is too broad, and
whether it introduces semantics the RTL does not show. A signal being a control signal does
not by itself establish a role such as "security control" or "privilege control".

## relationships
Verify BOTH the type and the target. Take the type literally:
  CAPTURES     — a value stored on a clock edge
  DERIVES_FROM — computed from
  GATES        — controls whether another operation occurs
  SELECTS      — used to choose among alternatives
  SOURCES      — drives / is assigned into (x <= y supports SOURCES from y to x)
  CONSTRAINS   — bounds or limits
  CARRIES      — conveys a value across
  SEQUENCES    — clock-driven sequencing of state
The parser may use relationship types outside this list. Audit any such type literally, by its
plain meaning, on the same evidentiary standard. A type lying outside the list above is NOT by
itself an error and must never be reported as one — judge only whether the RTL supports the
stated dependency between the source element and the target.

Do not infer a relationship type merely because two signals appear in the same process. A
reset that overrides a register must NOT automatically be classified as SEQUENCES — if the
RTL shows `if rstn_i = '0' then ctrl <= ...`, then a claim `rstn_i SEQUENCES ctrl` is using
SEQUENCES to mean ordinary clock-driven sequencing and should be flagged.

## relationship targets
A relationship is wrong if the target is wrong even when the type is right. For each target:
confirm it exists, locate the connection, and determine whether the stated dependency
actually holds. If the target is only affected indirectly through another signal, do not
silently treat the indirect chain as a direct relationship — report the distinction and use
PARTIALLY_SUPPORTED.

## evidence — stricter than the prose
Locate the cited construct, confirm it exists, and confirm it actually supports the claim.
Distinguish "evidence exists" from "evidence proves the claim". `bus_req_o <= host_req_i`
supports a pass-through; it does not prove every field is separately consumed or interpreted.
Citing a process that merely contains rstn_i does not establish that rstn_i sequences a
register. Record the RTL location and say whether the evidence is direct or only indirect.

## conditionals, generates, active configuration
Pay attention to if / elsif / case / when / generate / generic-dependent branches /
constants controlling implementation style / reset branches. Do not merge mutually exclusive
behaviors into one unconditional description. If a constant such as
`constant alt_style_c : boolean := false;` gates an `if alt_style_c generate` block, behavior
inside that block is NOT active in the default configuration, and describing it as
unconditional is PARTIALLY_SUPPORTED or CONTRADICTED depending on severity. Do not assume
every branch is simultaneously active.

## reset
Distinguish asynchronous reset, synchronous reset, reset-driven override, and ordinary
clocked state update. Do not classify reset behavior as ordinary sequencing.

## combinational vs registered
Distinguish combinational, registered, latched, asynchronous, directly assigned, and derived.
Do not call a combinational wire persistent state. Do not call a registered signal a
transient combinational value. Do not infer persistence merely from participation in a
multi-cycle process.

## cross-entity
If a claim crosses an entity boundary, verify the instance port map, the source port and the
destination port, and distinguish direct RTL evidence from downstream inference. Do not
invent downstream behavior of another entity unless that entity's RTL was supplied.

# Deterministic inputs — three kinds, three different weights
- CONFIRMED MECHANICAL PROBLEMS: structural facts with no semantics involved (a duplicate
  record key, an entity with no declaration in the RTL set). Reliable.
- DETERMINISTIC FACTS: measurements — assignment style, reset-guarded assignment, generate
  guards, and the tier at which each symbol resolved. Reliable as measurements. They carry no
  verdict. In particular a symbol resolution of NOT_ESTABLISHED_BY_STATIC_CHECK means the
  static matcher failed, not that the symbol is absent from the design.
- REVIEW HINTS: questions only. See above. No presumption of error.

You decide every status yourself, from the RTL.

# Error categories — use exactly these
FUNCTIONALITY_ERROR, ROLE_ERROR, RELATIONSHIP_ERROR, RELATIONSHIP_TARGET_ERROR,
EVIDENCE_ERROR, MISSING_BEHAVIOR, CONDITIONAL_BEHAVIOR_ERROR, RESET_BEHAVIOR_ERROR,
COMBINATIONAL_REGISTER_ERROR, CROSS_ENTITY_ERROR, UNVERIFIABLE_FILE_MAPPING,
UNVERIFIABLE_CLAIM

Severity: CRITICAL (directly contradicts RTL) / MAJOR (materially changes the behavioral
interpretation) / MINOR (imprecise or incomplete, behavior unchanged) / NONE.

# Output — strict JSON only, no prose, no markdown fence
{
  "module": "...", "entity": "...", "element": "...",
  "functionality_check": {"status": "...", "evidence_type": "...", "confidence": 0.0,
                          "parser_claim": "...", "rtl_evidence": "...", "rtl_location": "...",
                          "rtl_trace": [{"step": "...", "rtl_location": "..."}],
                          "interpretation": ""},
  "role_checks": [{"role": "...", "status": "...", "evidence_type": "...", "confidence": 0.0,
                   "rtl_evidence": "...", "rtl_location": "...", "rtl_trace": [], "interpretation": ""}],
  "relationship_checks": [{"type": "...", "target": "...", "status": "...", "evidence_type": "...",
                           "confidence": 0.0, "rtl_evidence": "...", "rtl_location": "...",
                           "rtl_trace": [], "interpretation": ""}],
  "evidence_check": {"status": "...", "evidence_type": "...", "confidence": 0.0,
                     "parser_evidence": "...", "rtl_evidence": "...", "rtl_location": "...",
                     "rtl_trace": [], "interpretation": ""},
  "overall_status": "...", "overall_confidence": 0.0,
  "errors": [{"error_type": "...", "field": "...", "claim": "...", "status": "...",
              "rtl_evidence": "...", "rtl_location": "...", "severity": "...", "explanation": "..."}]
}

rtl_location must cite line numbers from the supplied excerpt, e.g. "L412-L418".
"errors" is [] when nothing is wrong.

The payload carries an AUDIT CHECKLIST enumerating every check you must return. Return
EXACTLY one check per numbered line, in that order — no merging, no skipping. Counts that
do not match the checklist make the response incomplete.

In particular, a relationship whose target list is EMPTY still gets its own
relationship_check. Set `"target": ""` and audit the claim itself: a claim that an element
is ISOLATED, or otherwise stands in some relation to nothing, is a substantive assertion
about the RTL and is exactly as auditable as one naming a target. Do not omit it because
there is no target to look up. Never merge several targets into one check either — four
targets means four checks, each with its own status.

`status` and `evidence_type` are two DIFFERENT fields with two DIFFERENT vocabularies. Do not
mix them up:
  "status"        takes ONLY: SUPPORTED | PARTIALLY_SUPPORTED | CONTRADICTED |
                              NOT_ESTABLISHED | UNVERIFIABLE      (the verdict)
  "evidence_type" takes ONLY: DIRECT_RTL | TRACED_RTL | INTERPRETATION  (how you know)
Putting DIRECT_RTL/TRACED_RTL/INTERPRETATION in "status", or a verdict in "evidence_type",
makes the whole response invalid. Every check object needs both fields, each from its own
list.

"confidence" is a number in [0,1] recording how firmly the RTL settles that status. It is a
diagnostic, NOT a substitute for evidence: every status still needs rtl_evidence and
rtl_location regardless of confidence. Never raise confidence to compensate for weak evidence,
and never use it to soften a status you cannot support. A CONTRADICTED at 0.99 must still
quote the RTL that contradicts the claim.

"rtl_trace" is required and non-empty when evidence_type is TRACED_RTL, and [] otherwise.
"interpretation" is required and non-empty when evidence_type is INTERPRETATION — state
plainly what the judgement was — and "" otherwise.
""".strip()

VERIFIER_PROMPT_SHA = hashlib.sha256(VERIFIER_SYSTEM.encode()).hexdigest()
print("verifier prompt", VERIFIER_PROMPT_VERSION, "sha", VERIFIER_PROMPT_SHA[:16],
      "|", len(VERIFIER_SYSTEM), "chars")

# scope guard: the verifier prompt must not carry asset/security vocabulary as an instruction
for bad in ["back_test", "C/I/A/U asset", "is an asset", "Confidentiality:"]:
    assert bad not in VERIFIER_SYSTEM, f"scope leak in verifier prompt: {bad}"
print("scope guard: clean")

verifier prompt VP5 sha 8246f9f9f46b3a8d | 17650 chars
scope guard: clean


In [11]:
def audit_checklist(rec):
    """Enumerate EXACTLY the checks the verifier must return for this element.

    VP4 and earlier told the model "emit one relationship_check per (type, target)
    pair" and left it to infer the rest. A relationship with targets: [] has ZERO
    pairs, so the model correctly emitted nothing and the claim was never audited --
    678 of 2464 relationships corpus-wide (27.5%) are shaped that way. The same
    instruction also permitted merging several targets into one check.

    Enumerating removes the inference. -> (text, n_roles, n_rel_checks)
    """
    roles = rec.get("role") or []
    pairs = []
    for r in rec.get("relationship") or []:
        t = r.get("type")
        tgts = r.get("targets") or []
        if not tgts:
            pairs.append((t, None))          # no target named -- still ONE check
        else:
            pairs += [(t, x) for x in tgts]  # one check PER target, never merged
    # NOTE: do not write the schema field names ("functionality_check", "role_checks",
    # ...) here. assert_no_prior_results_used() treats verifier-output vocabulary in a
    # payload as an anti-drift violation, and it is right to -- that guard is what stops
    # a previous run's output being fed back in as evidence. The prompt already defines
    # which JSON field each check belongs to, so the checklist only has to enumerate
    # WHAT to check, not name the field.
    out = [f"functionality  -> exactly 1 check",
           f"roles          -> exactly {len(roles)} check(s)"]
    for i, r in enumerate(roles, 1):
        out.append(f"    R{i}. {r}")
    out.append(f"relationships  -> exactly {len(pairs)} check(s)")
    for i, (t, x) in enumerate(pairs, 1):
        if x is None:
            out.append(f'    L{i}. {t} -> (parser named NO target; return this check with'
                       f' "target": "" and audit whether the {t} claim itself holds)')
        else:
            out.append(f"    L{i}. {t} -> {x}")
    out.append("evidence       -> exactly 1 check")
    return "\n".join(out), len(roles), len(pairs)

def build_user_payload(rec, ctx, ctx_meta, det_findings, det_facts, det_hints=None):
    claims = {
        "module": rec["module"], "entity": rec["entity"], "element": rec["name"],
        "declared_type": rec.get("type"), "declared_dir": rec.get("dir"),
        "declared_kind": rec.get("kind"),
        "function": rec.get("function"),
        "functionality": rec.get("functionality"),
        "role": rec.get("role"),
        "relationship": rec.get("relationship"),
        "evidence": rec.get("evidence"),
    }
    mode = ctx_meta.get("context_mode", "CLAIM_DRIVEN_EXCERPT")
    if mode == "FULL_ARCHITECTURE":
        mode_note = [
            "CONTEXT MODE: FULL_ARCHITECTURE.",
            "This is the COMPLETE entity declaration and the COMPLETE architecture body,",
            "contiguous and unabridged. Nothing was filtered, and no line was selected",
            "because the parser mentioned it. ABSENCE IS EVIDENCE here: if a claimed",
            "signal, assignment, condition or dependency does not appear anywhere below,",
            "it does not exist in this entity.",
        ]
    else:
        mode_note = [
            "CONTEXT MODE: CLAIM_DRIVEN_EXCERPT.",
            "The architecture was too large to send whole, so this is a slice assembled by",
            "SEARCHING FOR THE SYMBOLS THE PARSER NAMED. It is therefore biased toward the",
            "parser's claim. ABSENCE FROM THIS SLICE IS NOT EVIDENCE OF ABSENCE -- use",
            "NOT_ESTABLISHED, and never CONTRADICTED, when something simply is not here.",
        ]
    parts = [
        # RTL FIRST. It is identical for every element of an entity, so putting it at
        # the front makes it a shared prompt prefix and lets provider-side prompt
        # caching cover it -- 582 of the 1689 elements share one architecture.
        f"## RTL — file {ctx_meta.get('rtl_file')}, "
        f"architecture lines {ctx_meta.get('arch_span')}, "
        f"truncated={ctx_meta.get('truncated')}",
        *mode_note,
        "Line numbers are 1-based from the original file. `...` marks elided regions.",
        "",
        ctx if ctx else "(NO RTL COULD BE MAPPED FOR THIS ENTITY)",
        "",
        "## PARSER CLAIMS (hypothesis to be checked)",
        json.dumps(claims, indent=2, ensure_ascii=False),
        "",
        "## AUDIT CHECKLIST — return exactly one check per numbered line, in this order.",
        "Do not merge two lines into one check. Do not skip a line. A relationship whose",
        "target list is empty still gets its own check — the claim is auditable even when",
        "the parser named nothing for it to point at.",
        audit_checklist(rec)[0],
        "",
        "## CONFIRMED MECHANICAL PROBLEMS (structural only, no semantics involved)",
        json.dumps(det_findings, indent=2, ensure_ascii=False) if det_findings else "[] (none)",
        "",
        "## DETERMINISTIC FACTS (measurements, no verdict attached)",
        json.dumps(det_facts, indent=2, ensure_ascii=False) if det_facts else "{}",
        "",
        "## REVIEW HINTS — these are QUESTIONS, NOT findings of error.",
        "A hint means a static check noticed something worth your attention. It is NOT",
        "evidence that the parser is wrong. Answer each question from the RTL above. If",
        "the RTL supports the parser, say SUPPORTED regardless of the hint.",
        json.dumps(det_hints or [], indent=2, ensure_ascii=False) if det_hints else "[] (none)",
        "",
        # REQUIRED by the Responses API: with text.format=json_object the literal
        # word "json" must appear in `input`. It is not enough for `instructions`
        # to say JSON -- the check runs against the input messages, and omitting
        # it fails every call with HTTP 400 before any tokens are billed.
        "Return your audit as a single json object exactly matching the schema given in the "
        "instructions. Output json only — no prose, no markdown fence.",
    ]
    return "\n".join(parts)

_payload = build_user_payload(_probe, _c, _m, _f, _facts, _hints)
assert_no_prior_results_used(_payload, "probe payload")
print(_payload[:900])
print("\n... payload chars:", len(_payload))

## RTL — file neorv32_boot_rom.vhd, architecture lines [27, 70], truncated=False
CONTEXT MODE: FULL_ARCHITECTURE.
This is the COMPLETE entity declaration and the COMPLETE architecture body,
contiguous and unabridged. Nothing was filtered, and no line was selected
because the parser mentioned it. ABSENCE IS EVIDENCE here: if a claimed
signal, assignment, condition or dependency does not appear anywhere below,
it does not exist in this entity.
Line numbers are 1-based from the original file. `...` marks elided regions.

   18 | 
   19 | entity neorv32_boot_rom is
   20 |   port (
   21 |     clk_i     : in  std_ulogic; -- global clock line
   22 |     rstn_i    : in  std_ulogic; -- async reset, low-active
   23 |     bus_req_i : in  bus_req_t;  -- bus request
   24 |     bus_rsp_o : out bus_rsp_t   -- bus response
   25 |   );
   26 | end neorv32_boot_rom;
   27 | 
   28 | architecture neo

... payload chars: 7589


## 7. API client and response validation

Matches the client used for generation: OpenAI **Responses API** with `instructions` /
`input`, `reasoning={"effort": ...}`, `max_output_tokens`, JSON-object output, per-call token
accounting and a USD cost report.

**No temperature, no seed.** Reasoning models reject both. The knob is `reasoning_effort`, and
run-to-run stability comes from the response cache — keyed on
`sha256(provider + model + effort + reasoning_flag + max_tokens + prompt_sha + payload)`.
The manifest says this outright rather than implying determinism the API cannot give.

**Budget escalation.** For a reasoning model the reasoning trace shares `max_output_tokens`
with the answer, so a verbose schema can consume the whole budget and return empty text. That
is a budget problem, not a model failure, and booking it as `parse_error` would corrupt the
parse-error rate. On `status == "incomplete"` the call retries at double the budget up to
`max_output_tokens_ceiling`; escalations, and anything still incomplete at the ceiling, are
counted and reported.

`max_output_tokens` defaults to 6000 rather than something smaller because this schema emits a
check per role and per relationship target, each with `rtl_evidence`, `rtl_location`,
`rtl_trace` and `interpretation`.

Failures are recorded, never silently skipped. Malformed JSON is stored raw and marked
`parse_error` so genuine parse failures stay measurable.

In [12]:
class TransientError(Exception): pass
STATUSES = {"SUPPORTED","PARTIALLY_SUPPORTED","CONTRADICTED","NOT_ESTABLISHED","UNVERIFIABLE"}
ERROR_TYPES = {"FUNCTIONALITY_ERROR","ROLE_ERROR","RELATIONSHIP_ERROR","RELATIONSHIP_TARGET_ERROR",
               "EVIDENCE_ERROR","MISSING_BEHAVIOR","CONDITIONAL_BEHAVIOR_ERROR","RESET_BEHAVIOR_ERROR",
               "COMBINATIONAL_REGISTER_ERROR","CROSS_ENTITY_ERROR","UNVERIFIABLE_FILE_MAPPING",
               "UNVERIFIABLE_CLAIM"}
SEVERITIES = {"CRITICAL","MAJOR","MINOR","NONE"}
EVIDENCE_TYPES = {"DIRECT_RTL","TRACED_RTL","INTERPRETATION"}

def _cache_key(payload, cfg):
    """
    Cache identity for one verifier call.

    run_tag is intentionally included so that an explicit replicate
    can bypass a previous cached response without changing the prompt,
    model, or decoding settings.
    """
    h = hashlib.sha256()

    parts = [
        cfg["provider"],
        cfg["model"],
        str(cfg["reasoning_effort"]),
        str(USE_REASONING),
        str(cfg["max_output_tokens"]),
        str(cfg.get("run_tag", "")),
        str(cfg.get("verifier_prompt_version", "")),
        str(cfg.get("parsed_data_version", "")),
        str(cfg.get("rtl_dataset_version", "")),
        VERIFIER_PROMPT_SHA,
        payload,
    ]

    for p in parts:
        h.update(p.encode("utf-8"))
        h.update(b"\x00")

    return h.hexdigest()

USAGE = []          # per-call token accounting. MUST exist before any API call:
                    # _call() appends to it and cost() reads it.

# USAGE accumulates for the whole KERNEL SESSION and is never reset -- that is
# deliberate, it is the session total. But a manifest must report ONE RUN, so
# run_verification() records the slice belonging to its own run here and save_run()
# bills against that. Without this, running the STAGE cell twice in one kernel
# makes the second manifest report the sum of both runs as if it were the second.
LAST_RUN_USAGE = None

def _call(system, user, cfg, max_tokens=None):
    max_tokens = max_tokens or cfg["max_output_tokens"]
    if cfg["provider"] == "openai":
        from openai import OpenAI, APITimeoutError, APIConnectionError, RateLimitError, InternalServerError
        cl = OpenAI(api_key=API_KEY, timeout=cfg["request_timeout_s"])
        kw = dict(model=cfg["model"], instructions=system, input=user,
                  max_output_tokens=max_tokens,
                  text={"format": {"type": "json_object"}})
        if USE_REASONING:
            kw["reasoning"] = {"effort": cfg["reasoning_effort"]}
        try:
            r = cl.responses.create(**kw)
        except (APITimeoutError, APIConnectionError, RateLimitError, InternalServerError) as e:
            raise TransientError(f"{type(e).__name__}: {e}")
        u = r.usage
        rt = getattr(getattr(u, "output_tokens_details", None), "reasoning_tokens", 0) or 0
        ci = getattr(getattr(u, "input_tokens_details", None), "cached_tokens", 0) or 0
        usage = {"model": cfg["model"], "in": u.input_tokens, "out": u.output_tokens,
                 "reasoning": rt, "cached_in": ci,
                 "incomplete": getattr(r, "status", None) == "incomplete",
                 "incomplete_reason": getattr(getattr(r, "incomplete_details", None),
                                              "reason", None),
                 "budget": max_tokens}
        USAGE.append(usage)
        return (r.output_text or ""), usage
    elif cfg["provider"] == "anthropic":
        import anthropic
        cl = anthropic.Anthropic(api_key=API_KEY, timeout=cfg["request_timeout_s"])
        try:
            r = cl.messages.create(model=cfg["model"], system=system,
                                   messages=[{"role":"user","content":user}],
                                   temperature=cfg["temperature"],
                                   max_tokens=cfg["max_output_tokens"])
        except (anthropic.APITimeoutError, anthropic.APIConnectionError,
                anthropic.RateLimitError, anthropic.InternalServerError) as e:
            raise TransientError(f"{type(e).__name__}: {e}")
        usage = {"model": cfg["model"], "in": r.usage.input_tokens,
                 "out": r.usage.output_tokens, "reasoning": 0, "cached_in": 0,
                 "incomplete": False, "incomplete_reason": None, "budget": max_tokens}
        USAGE.append(usage)
        return "".join(b.text for b in r.content if getattr(b,"type","")=="text"), usage
    raise ValueError(f"unknown provider {cfg['provider']!r}")

def call_with_retry(system, user, cfg):
    """Retries transient errors with exponential backoff, and separately escalates
    the output budget when the model ran out of tokens mid-answer. For a reasoning
    model the reasoning trace shares `max_output_tokens`, so a verbose schema can
    consume the whole budget and return empty text -- that is a budget problem, not
    a model failure, and must not be recorded as a parse error."""
    last, budget, escalations = None, cfg["max_output_tokens"], 0
    for a in range(cfg["max_retries"]):
        try:
            t0 = time.time(); txt, usage = _call(system, user, cfg, max_tokens=budget)
            if usage.get("incomplete") and cfg["escalate_on_incomplete"] \
               and budget < cfg["max_output_tokens_ceiling"]:
                budget = min(budget * 2, cfg["max_output_tokens_ceiling"])
                escalations += 1
                print(f"    [llm warn] incomplete ({usage.get('incomplete_reason')}) "
                      f"-> retrying at max_output_tokens={budget}")
                continue
            # `a` counts loop iterations, and an escalation consumes one. Subtract
            # them so `retries` means what it says: TRANSIENT API failures retried.
            # An escalation is not a retry -- the API succeeded, the answer did not fit.
            return {"text": txt, "usage": usage, "latency_s": round(time.time()-t0,3),
                    "retries": a - escalations, "escalations": escalations, "error": None,
                    "incomplete": bool(usage.get("incomplete"))}
        except TransientError as e:
            last = str(e)
            time.sleep(min(cfg["backoff_cap_s"], cfg["backoff_base_s"]*(2**a))*(0.5+random.random()/2))
        except Exception as e:
            return {"text":"","usage":{},"latency_s":0.0,"retries":a,
                    "escalations":escalations,"error":f"{type(e).__name__}: {e}",
                    "incomplete":False}
    return {"text":"","usage":{},"latency_s":0.0,"retries":cfg["max_retries"],
            "escalations":escalations,"incomplete":False,
            "error":f"exhausted retries: {last}"}

def cost(cfg=CONFIG, usage=None):
    """USD for calls so far, at per-1M-token prices."""
    us = USAGE if usage is None else usage
    i  = sum(u["in"] - u.get("cached_in", 0) for u in us)
    c  = sum(u.get("cached_in", 0) for u in us)
    o  = sum(u["out"] for u in us)
    rt = sum(u.get("reasoning", 0) for u in us)
    total = (i*cfg["price_in"] + c*cfg["price_cached_in"] + o*cfg["price_out"]) / 1e6
    return {"calls": len(us), "in": i, "cached_in": c, "out": o, "reasoning": rt,
            "incomplete": sum(1 for u in us if u.get("incomplete")),
            "usd": round(total, 4)}

def escalation_waste(cfg=CONFIG, usage=None):
    """USD spent on truncated answers that were discarded and re-asked.

    An escalation re-issues the call at a bigger budget. The first, truncated
    response was still generated and still billed at the output rate, then thrown
    away. cost() counts it (correctly -- the money is gone) but nothing separated
    it from useful spend, so the waste was invisible.

    Note max_output_tokens is a CAP, not a charge: across this project's 99 cached
    calls the median output was 4 384 tokens against a 12 000 budget and nothing
    was ever billed at the cap. Raising the START budget therefore costs nothing
    for elements that do not need it, and removes this waste for those that do."""
    us = USAGE if usage is None else (usage or [])
    dead = [u for u in us if u.get("incomplete")]
    out = sum(u.get("out", 0) for u in dead)
    # the discarded call billed its INPUT too -- the whole payload was sent and
    # read before the answer was cut off. Counting only output understates it.
    cin = sum(u.get("cached_in", 0) for u in dead)
    uin = sum(u.get("in", 0) - u.get("cached_in", 0) for u in dead)
    usd = (out * cfg["price_out"] + uin * cfg["price_in"]
           + cin * cfg["price_cached_in"]) / 1e6
    return {"discarded_calls": len(dead), "discarded_output_tokens": out,
            "discarded_input_tokens": uin + cin,
            "wasted_usd": round(usd, 4)}

def project_cost(n_elements, cfg=CONFIG, n_elements_done=None):
    """Extrapolate a full run from the pilot.

    Divide by ELEMENTS COMPLETED, not by calls. A budget escalation issues a
    second and third call for the SAME element, so usd/call sits systematically
    below usd/element whenever escalation fires, and projecting on usd/call
    understates the full run. Pass n_elements_done (e.g. len(RESULTS)); it falls
    back to the call count only when nothing better is known."""
    if not USAGE: return {"note": "no calls yet"}
    c = cost(cfg)
    done = n_elements_done or c["calls"]
    per_el   = c["usd"] / max(1, done)
    per_call = c["usd"] / max(1, c["calls"])
    return {"calls_so_far": c["calls"], "elements_done": done,
            "usd_so_far": c["usd"],
            "usd_per_element": round(per_el, 5),
            "usd_per_call": round(per_call, 5),
            "calls_per_element": round(c["calls"] / max(1, done), 3),
            f"projected_usd_for_{n_elements}": round(per_el * n_elements, 2)}

def validate_verification(raw):
    """Strict. No silent repair."""
    if not raw or not raw.strip(): return None, "parse_error", "empty response"
    t = raw.strip()
    if t.startswith("```"): t = re.sub(r"^```(?:json)?\s*|\s*```$","",t,flags=re.S).strip()
    try: o = json.loads(t)
    except Exception as e: return None, "parse_error", f"json decode: {e}"
    if not isinstance(o, dict): return None, "parse_error", "top level not an object"
    for k in ["functionality_check","evidence_check"]:
        if not isinstance(o.get(k), dict): return None, "parse_error", f"missing {k}"
        if o[k].get("status") not in STATUSES:
            return None, "parse_error", f"{k}.status={o[k].get('status')!r}"
    for k in ["role_checks","relationship_checks","errors"]:
        if not isinstance(o.get(k), list): return None, "parse_error", f"{k} not a list"
    for c in o["role_checks"] + o["relationship_checks"]:
        if not isinstance(c, dict) or c.get("status") not in STATUSES:
            return None, "parse_error", f"bad check status {c!r}"[:120]
    if o.get("overall_status") not in STATUSES:
        return None, "parse_error", f"overall_status={o.get('overall_status')!r}"
    # evidence_type IS part of the audit contract -- it separates fact from the
    # verifier's own judgement -- so it is enforced strictly, no repair.
    missing_conf = []
    for label, node in ([("functionality", o["functionality_check"]),
                         ("evidence", o["evidence_check"])]
                        + [(f"role[{i}]", c) for i, c in enumerate(o["role_checks"])]
                        + [(f"rel[{i}]", c) for i, c in enumerate(o["relationship_checks"])]):
        et = node.get("evidence_type")
        if et not in EVIDENCE_TYPES:
            return None, "parse_error", f"{label}.evidence_type={et!r} not in {sorted(EVIDENCE_TYPES)}"
        tr = node.get("rtl_trace", [])
        if tr is None: tr = []
        if not isinstance(tr, list):
            return None, "parse_error", f"{label}.rtl_trace is not a list"
        for st in tr:
            if not isinstance(st, dict) or "rtl_location" not in st:
                return None, "parse_error", f"{label}.rtl_trace step missing rtl_location"
        # Strict maximum of 3 dependency hops.
        # Multiple RTL locations supporting ONE hop belong inside that hop's
        # rtl_location string and do not create additional trace steps.
        if len(tr) > 3:
            return None, "parse_error", (
                f"{label}.rtl_trace has {len(tr)} hops; hard maximum is 3"
            )
        if et == "TRACED_RTL" and not tr:
            return None, "parse_error", f"{label}.evidence_type=TRACED_RTL but rtl_trace is empty"
        if et == "INTERPRETATION" and not str(node.get("interpretation", "")).strip():
            return None, "parse_error", f"{label}.evidence_type=INTERPRETATION but interpretation is empty"
        node["rtl_trace"] = tr
        # confidence is a diagnostic, not contract. Validate when present; record
        # absence explicitly rather than discarding an otherwise valid audit.
        c = node.get("confidence")
        if c is None:
            missing_conf.append(label); continue
        if not isinstance(c, (int, float)) or not (0.0 <= float(c) <= 1.0):
            return None, "parse_error", f"{label}.confidence={c!r} not a number in [0,1]"
        node["confidence"] = float(c)
    oc = o.get("overall_confidence")
    if oc is not None and (not isinstance(oc, (int, float)) or not (0.0 <= float(oc) <= 1.0)):
        return None, "parse_error", f"overall_confidence={oc!r} not a number in [0,1]"
    if oc is None: missing_conf.append("overall")
    o["_confidence_missing"] = missing_conf
    for e in o["errors"]:
        if not isinstance(e, dict): return None, "parse_error", "error entry not an object"
        if e.get("error_type") not in ERROR_TYPES:
            return None, "parse_error", f"error_type={e.get('error_type')!r}"
        if e.get("severity") not in SEVERITIES:
            return None, "parse_error", f"severity={e.get('severity')!r}"
        if e.get("status") not in STATUSES:
            return None, "parse_error", f"error.status={e.get('status')!r}"
    return o, "ok", ""
print("client ready | provider:", CONFIG["provider"], "| model:", CONFIG["model"],
      "| reasoning:", USE_REASONING)
print("validator ready | statuses:", len(STATUSES), "| evidence types:", sorted(EVIDENCE_TYPES))

client ready | provider: openai | model: gpt-5-mini | reasoning: True
validator ready | statuses: 5 | evidence types: ['DIRECT_RTL', 'INTERPRETATION', 'TRACED_RTL']


## 8. Verification runner

In [13]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def claim_coverage(rec, v):
    """Did the verifier emit one check per parser claim?

    The prompt requires one relationship_check per (type, target) pair. The
    model sometimes merges several targets into a single check ("mem_ram_b0,
    mem_ram_b1, ..."), which silently drops claims from the audit and from every
    downstream count. This measures the loss; it does not repair it."""
    v = v or {}
    exp_roles = len(rec.get("role") or [])
    got_roles = len(v.get("role_checks") or [])
    exp_rels  = sum(max(1, len(x.get("targets") or []))
                    for x in (rec.get("relationship") or []))
    got_rels  = len(v.get("relationship_checks") or [])
    return {"roles_expected": exp_roles, "roles_returned": got_roles,
            "rel_pairs_expected": exp_rels, "rel_checks_returned": got_rels,
            "complete": (exp_roles == got_roles and exp_rels == got_rels)}

def verify_element(rec, cfg=CONFIG):
    ctx, meta = rtl_context_for(rec, cfg)
    det = DET.get(rec["element_id"]) or {}
    findings, facts = det.get("findings", []), det.get("facts", {})
    hints = det.get("hints", [])

    # entity unmappable -> deterministic UNVERIFIABLE, no API call
    if meta["mapping"] != "MAPPED":
        return {"element_id": rec["element_id"], "module": rec["module"],
                "entity": rec["entity"], "element": rec["name"],
                "rtl_file": None, "parse_status": "ok", "parse_detail": "",
                "cached": False, "api_error": None, "latency_s": 0.0, "retries": 0,
                "verification": {"module": rec["module"], "entity": rec["entity"],
                    "element": rec["name"],
                    "functionality_check": {"status":"UNVERIFIABLE","evidence_type":"DIRECT_RTL",
                                            "confidence":0.0,"parser_claim":rec.get("functionality"),
                                            "rtl_evidence":"","rtl_location":"","rtl_trace":[],
                                            "interpretation":""},
                    "role_checks": [{"role":r,"status":"UNVERIFIABLE","evidence_type":"DIRECT_RTL",
                                     "confidence":0.0,"rtl_evidence":"","rtl_location":"",
                                     "rtl_trace":[],"interpretation":""} for r in rec["role"]],
                    "relationship_checks": [{"type":x.get("type"),"target":t,"status":"UNVERIFIABLE",
                                             "evidence_type":"DIRECT_RTL","confidence":0.0,
                                             "rtl_evidence":"","rtl_location":"","rtl_trace":[],
                                             "interpretation":""}
                                            for x in rec["relationship"] for t in (x.get("targets") or [])],
                    "evidence_check": {"status":"UNVERIFIABLE","evidence_type":"DIRECT_RTL",
                                       "confidence":0.0,"parser_evidence":rec.get("evidence"),
                                       "rtl_evidence":"","rtl_location":"","rtl_trace":[],
                                       "interpretation":""},
                    "overall_status": "UNVERIFIABLE",
                    "errors": [{"error_type":"UNVERIFIABLE_FILE_MAPPING","field":"entity",
                                "claim":rec["entity"],"status":"UNVERIFIABLE","rtl_evidence":"",
                                "rtl_location":"","severity":"NONE",
                                "explanation":"No `entity <name> is` declaration found in the RTL dataset; not guessed."}]},
                "deterministic_findings": findings,
                "deterministic_facts": facts, "deterministic_hints": hints,
                "run_tag": cfg.get("run_tag"),
                "verifier_prompt_version": cfg.get("verifier_prompt_version"),
                "claim_coverage": {"roles_expected": len(rec.get("role") or []),
                                   "roles_returned": len(rec.get("role") or []),
                                   "rel_pairs_expected": 0, "rel_checks_returned": 0,
                                   "complete": True},
                "raw_response": ""}

    payload = build_user_payload(rec, ctx, meta, findings, facts, hints)
    assert_no_prior_results_used(payload, f"payload for {rec['element_id']}")
    key = _cache_key(payload, cfg)
    cp = CACHE_DIR / key[:2] / (key + ".json")
    if cfg["cache_enabled"] and cp.exists():
        # A cache entry written when the kernel was interrupted mid-write is truncated
        # JSON. Over an 8-hour run that is a real possibility, and an unguarded
        # json.loads would turn one corrupt file into a permanent failure for that
        # element on every future run. Treat unreadable as absent and re-call.
        try:
            d = json.loads(cp.read_text(encoding="utf-8"))
            d["cached"] = True
            return d
        except Exception as e:
            print(f"    [cache unreadable, re-calling] {rec['module']}/{rec['name']}: "
                  f"{type(e).__name__}")

    res = call_with_retry(VERIFIER_SYSTEM, payload, cfg)
    parsed, status, detail = validate_verification(res["text"])
    row = {"element_id": rec["element_id"], "module": rec["module"], "entity": rec["entity"],
           "element": rec["name"], "rtl_file": meta.get("rtl_file"),
           "parse_status": status, "parse_detail": detail, "cached": False,
           "api_error": res["error"], "latency_s": res["latency_s"], "retries": res["retries"],
           "escalations": res.get("escalations", 0), "incomplete": res.get("incomplete", False),
           "usage": res.get("usage", {}),
           "verification": parsed, "deterministic_findings": findings,
           # provenance: what the verifier was actually shown, and which
           # replicate this row belongs to. Without these a saved run cannot be
           # reconstructed or attributed to a replicate.
           "deterministic_facts": facts, "deterministic_hints": hints,
           "run_tag": cfg.get("run_tag"),
           "verifier_prompt_version": cfg.get("verifier_prompt_version"),
           # WHICH RETRIEVAL MODE produced this verdict. FULL_ARCHITECTURE means
           # absence was evidence; CLAIM_DRIVEN_EXCERPT means it was not. Without
           # this a saved run cannot be interpreted, let alone compared.
           "context_mode": meta.get("context_mode"),
           "context_chars": len(ctx),
           "claim_coverage": claim_coverage(rec, parsed),
           "raw_response": res["text"]}
    # Cache RESULTS, not failures. A parse_error is a real observation about the
    # model's output and is cached so the rate stays measurable. An API-level
    # failure (4xx/5xx, exhausted retries) is not a result -- caching it would
    # freeze a transient or configuration fault permanently into the run.
    #
    # A response still INCOMPLETE at max_output_tokens_ceiling is the same kind of
    # thing as an API failure: the model did not finish because the BUDGET ran out,
    # not because it produced bad output. call_with_retry's docstring already says
    # such a case "must not be recorded as a parse error" -- but it reaches here
    # with error=None, validate_verification marks the truncated JSON parse_error,
    # and caching it would freeze a config fault in permanently: the element would
    # never be retried, not even after raising the ceiling. So it is NOT cached.
    if res["error"] is None and not res.get("incomplete"):
        cp.parent.mkdir(parents=True, exist_ok=True)
        cp.write_text(json.dumps(row, ensure_ascii=False), encoding="utf-8")
    elif res.get("incomplete"):
        print(f"    [not cached] {rec['module']}/{rec['name']}: still INCOMPLETE at the "
              f"{cfg['max_output_tokens_ceiling']} ceiling after "
              f"{res.get('escalations', 0)} escalation(s). Raise "
              f"max_output_tokens_ceiling, or lower reasoning_effort, then re-run: "
              f"this element will be re-called rather than replayed.")
    else:
        print(f"    [not cached] {rec['module']}/{rec['name']}: {str(res['error'])[:110]}")
    return row

def run_verification(records=None, cfg=CONFIG, limit=None, show_every=25):
    global LAST_RUN_USAGE
    recs = records if records is not None else PARSED
    if limit: recs = recs[:limit]
    usage_mark = len(USAGE)      # bill this run on USAGE[usage_mark:], not on all of it
    out, done, crashed = [], 0, []
    with ThreadPoolExecutor(max_workers=cfg["parallel_workers"]) as ex:
        futs = {ex.submit(verify_element, r, cfg): r for r in recs}
        for f in as_completed(futs):
            rec = futs[f]
            # An unguarded f.result() re-raises inside the loop, so ONE bad element
            # would abort the whole pass and discard every result collected so far.
            # The API responses are already cached, so nothing is re-paid for, but
            # the run dies and RESULTS is never assigned. Record and carry on.
            try:
                out.append(f.result())
            except Exception as e:
                crashed.append((rec, e))
                out.append({"element_id": rec["element_id"], "module": rec["module"],
                            "entity": rec["entity"], "element": rec["name"],
                            "rtl_file": None, "parse_status": "harness_error",
                            "parse_detail": f"{type(e).__name__}: {e}",
                            "cached": False, "api_error": None, "latency_s": 0.0,
                            "retries": 0, "escalations": 0, "incomplete": False,
                            "usage": {}, "verification": None,
                            "deterministic_findings": [], "deterministic_facts": {},
                            "deterministic_hints": [], "run_tag": cfg.get("run_tag"),
                            "verifier_prompt_version": cfg.get("verifier_prompt_version"),
                            "claim_coverage": {"complete": False,
                                               "roles_expected": 0, "roles_returned": 0,
                                               "rel_pairs_expected": 0,
                                               "rel_checks_returned": 0},
                            "raw_response": ""})
            done += 1
            if done % show_every == 0: print(f"  {done}/{len(recs)}", flush=True)
    LAST_RUN_USAGE = USAGE[usage_mark:]
    ok  = sum(1 for o in out if o["parse_status"] == "ok")
    esc = sum(o.get("escalations", 0) for o in out)
    inc = sum(1 for o in out if o.get("incomplete"))
    api = sum(1 for o in out if o.get("api_error"))
    if crashed:
        print(f"  !! {len(crashed)} element(s) raised inside the worker and were "
              f"recorded as harness_error rather than aborting the run:")
        for rec, e in crashed[:5]:
            print(f"       {rec['module']}/{rec['name']}: {type(e).__name__}: {e}")
    if api:
        print(f"  !! {api} element(s) failed at the API level (not cached; re-run to retry)")
    thin = [o for o in out if not (o.get("claim_coverage") or {}).get("complete", True)]
    print(f"verified {len(out)} | parsed_ok {ok/max(1,len(out)):.1%} | "
          f"cached {sum(1 for o in out if o.get('cached'))} | "
          f"budget escalations {esc} | still incomplete {inc}")
    if thin:
        lost = sum((o["claim_coverage"]["rel_pairs_expected"]
                    - o["claim_coverage"]["rel_checks_returned"]) for o in thin)
        print(f"  !! CLAIM COVERAGE: {len(thin)}/{len(out)} element(s) returned fewer "
              f"checks than the parser made claims; {lost} (type,target) pair(s) were "
              f"merged away and are NOT in any count below.")
        for o in thin[:8]:
            c = o["claim_coverage"]
            print(f"       {o['module']}/{o['element']}: roles "
                  f"{c['roles_expected']}->{c['roles_returned']}, rel pairs "
                  f"{c['rel_pairs_expected']}->{c['rel_checks_returned']}")
    if inc:
        print(f"  !! {inc} response(s) hit the {cfg['max_output_tokens_ceiling']} token "
              f"ceiling and were NOT cached. Raise max_output_tokens_ceiling or lower "
              f"reasoning_effort, then re-run -- they will be re-called, not replayed.")
    if esc:
        w = escalation_waste(cfg, usage=LAST_RUN_USAGE)
        print(f"  escalation cost: {w['discarded_calls']} truncated answer(s) billed and "
              f"discarded = {w['discarded_output_tokens']} output + "
              f"{w['discarded_input_tokens']} input tokens = ${w['wasted_usd']}. "
              f"max_output_tokens is a CAP, not a charge -- raising it costs nothing "
              f"for elements that fit. See the CONFIG note.")
    print("  this run  :", json.dumps(cost(cfg, usage=LAST_RUN_USAGE)))
    if len(LAST_RUN_USAGE) != len(USAGE):
        print("  session   :", json.dumps(cost(cfg)),
              "  <- cumulative across every run in this kernel")
    print("  run_tag:", cfg.get("run_tag"), "| verifier:",
          cfg.get("verifier_prompt_version"), "| prompt sha:", VERIFIER_PROMPT_SHA[:16])
    return out
print("run_verification() ready")

run_verification() ready


### Reset-hint distribution

512 reset hints only matters if they sit on the claims that are actually interesting. The
section-12 case is a **reset signal claimed to SEQUENCE a register**; a reset assignment
merely appearing somewhere downstream of an ordinary signal is far weaker. This splits them.

In [14]:
def reset_hint_distribution():
    rows = []
    for r in PARSED:
        d = DET.get(r["element_id"]) or {}
        if not any(h["hint"] == "target_assigned_in_reset_branch" for h in d.get("hints", [])):
            continue
        src_reset = bool(re.search(r"(?i)(^|_)(rstn?|reset)(_|$|\d)", str(r["name"])))
        for rel in r["relationship"]:
            t = str(rel.get("type", "")).upper()
            for tgt in (rel.get("targets") or []):
                info = ((d.get("facts") or {}).get("targets") or {}).get(str(tgt)) or {}
                if not info.get("assigned_under_reset"): continue
                if src_reset and t == "SEQUENCES":   cat = "reset source -> SEQUENCES"
                elif src_reset:                      cat = f"reset source -> {t}"
                elif t == "SEQUENCES":               cat = "non-reset source -> SEQUENCES"
                elif t == "CAPTURES":                cat = "non-reset source -> CAPTURES"
                else:                                cat = f"non-reset source -> {t}"
                rows.append({"category": cat, "module": r["module"], "entity": r["entity"],
                             "element": r["name"], "type": t, "target": str(tgt)})
    df_ = pd.DataFrame(rows)
    if not len(df_):
        print("no reset hints"); return df_
    summ = df_.groupby("category").size().reset_index(name="count").sort_values("count", ascending=False)
    print("reset-related review hints by category:")
    for _, x in summ.iterrows(): print("   %-40s %5d" % (x["category"], x["count"]))
    print()
    top = df_[df_.category == "reset source -> SEQUENCES"]
    print(f"the genuinely interesting category (reset source -> SEQUENCES): {len(top)}")
    for _, x in top.head(8).iterrows():
        print(f"   {x['module']}/{x['entity']}/{x['element']} SEQUENCES -> {x['target']}")
    return df_

RESET_DIST = reset_hint_distribution()

reset-related review hints by category:
   non-reset source -> SEQUENCES              161
   non-reset source -> CAPTURES               150
   reset source -> SEQUENCES                  149
   non-reset source -> SOURCES                 61
   reset source -> OVERRIDES                   20
   non-reset source -> GATES                   18
   non-reset source -> SELECTS                 17
   reset source -> GATES                       16
   non-reset source -> CONSTRAINS              15
   reset source -> DERIVES_FROM                10
   reset source -> SOURCES                      6
   non-reset source -> SLICES                   4
   non-reset source -> OVERRIDES                1

the genuinely interesting category (reset source -> SEQUENCES): 149
   neorv32_bus/neorv32_bus_switch/rstn_i SEQUENCES -> state
   neorv32_bus/neorv32_bus_switch/rstn_i SEQUENCES -> sel_q
   neorv32_bus/neorv32_bus_switch/rstn_i SEQUENCES -> locked
   neorv32_bus/neorv32_bus_switch/rstn_i SEQUENCES -> a_req


## 8b. Curated smoke set — inspect before spending 1689 calls

The point of this sample is **not** to measure accuracy. It is to check that the verifier
behaves like an auditor rather than a second parser or an error generator. Read the four
columns together: parser claim → deterministic facts → the verifier's RTL evidence → its
status and reasoning.

The set is chosen to cover the cases most likely to expose bad behaviour:

| case | why it is in the set |
|---|---|
| a reset port (`rstn_i`) | the section-12 `SEQUENCES` trap |
| a generate-dependent element (`addr_ff`, `alt_style_c=false`) | inactive-branch-as-active |
| an ordinary data/address field | the baseline; over-flagging shows up here |
| a record field resolving via the package | tests the resolution hierarchy |
| a relationship using a non-spec type | must not be called an error for that reason |
| an element with an unresolved evidence citation | must yield NOT_ESTABLISHED, not CONTRADICTED |

In [15]:
def build_smoke_set():
    picked, seen = [], set()
    def take(rec, why):
        if rec is None or rec["element_id"] in seen: return
        seen.add(rec["element_id"]); picked.append((why, rec))
    def first(pred):
        return next((r for r in PARSED if pred(r)), None)

    take(first(lambda r: re.fullmatch(r"(?i)rstn?_i", str(r["name"])) and
               any(str(x.get("type","")).upper() == "SEQUENCES" for x in r["relationship"])),
         "reset port with a SEQUENCES claim")
    take(first(lambda r: str(r["name"]) == "addr_ff"), "generate-dependent element")
    take(first(lambda r: any((DET.get(r["element_id"]) or {}).get("hints", []) and
               h["hint"] == "target_in_generate_branch"
               for h in (DET.get(r["element_id"]) or {}).get("hints", []))),
         "target under a generate guard")
    take(first(lambda r: str(r["name"]).endswith(".addr") and r["relationship"]),
         "ordinary address field (baseline)")
    take(first(lambda r: any(h["hint"] == "evidence_citation_resolved_elsewhere"
               for h in (DET.get(r["element_id"]) or {}).get("hints", []))),
         "citation resolving via package / other entity")
    take(first(lambda r: any(str(x.get("type","")).upper() not in SPEC_TYPES
               for x in r["relationship"])), "relationship using a non-spec type")
    take(first(lambda r: any(h["hint"] == "evidence_citation_unresolved_by_static_check"
               for h in (DET.get(r["element_id"]) or {}).get("hints", []))),
         "unresolved evidence citation (must be NOT_ESTABLISHED, not CONTRADICTED)")
    take(first(lambda r: not r["relationship"]), "element with no relationships")
    return picked

SMOKE_SET = build_smoke_set()
print(f"curated smoke set: {len(SMOKE_SET)} elements\n")
for why, r in SMOKE_SET:
    print(f"  {why:56s} {r['module']}/{r['entity']}/{r['name']}")

curated smoke set: 8 elements

  reset port with a SEQUENCES claim                        neorv32_bus/neorv32_bus_switch/rstn_i
  generate-dependent element                               neorv32_imem/neorv32_imem/addr_ff
  target under a generate guard                            neorv32_bus/neorv32_bus_reg/clk_i
  ordinary address field (baseline)                        neorv32_boot_rom/neorv32_boot_rom/bus_req_i.addr
  citation resolving via package / other entity            neorv32_boot_rom/neorv32_boot_rom/bus_req_i.data
  relationship using a non-spec type                       neorv32_boot_rom/neorv32_boot_rom/rstn_i
  unresolved evidence citation (must be NOT_ESTABLISHED, not CONTRADICTED) neorv32_bus/neorv32_bus_io_switch/dev_req
  element with no relationships                            neorv32_cpu_pmp/neorv32_cpu_pmp/ctrl_i.if_fence


In [16]:
# ---- run the curated smoke set (one API call each) ----------------------------
# COMMENTED OUT after a clean 8/8 pass. Re-enable only if the verifier prompt,
# the payload builder, or the RTL context extractor changes -- any of those
# invalidates the cache and the smoke set is the cheapest way to re-validate.
SMOKE_ENABLED = False

if not SMOKE_ENABLED:
    print("smoke set disabled (SMOKE_ENABLED = False). Last result: 8/8 parsed, "
          "0 API errors, 0 escalations, max trace depth 3.")
    print(f"{len(SMOKE_SET)} elements are selected and ready if you re-enable it.")
else:
    SMOKE = run_verification([r for _, r in SMOKE_SET], show_every=1)

def inspect(results, smoke_set):
    idx = {r["element_id"]: (why, r) for why, r in smoke_set}
    for res in results:
        why, rec = idx.get(res["element_id"], ("", {}))
        v = res.get("verification") or {}
        print("=" * 100)
        print(f"[{why}]  {res['module']}/{res['entity']}/{res['element']}   file={res['rtl_file']}")
        print(f"  PARSER functionality : {str(rec.get('functionality'))[:190]}")
        print(f"  PARSER roles         : {rec.get('role')}")
        print(f"  PARSER relationships : {rec.get('relationship')}")
        print(f"  PARSER evidence      : {str(rec.get('evidence'))[:190]}")
        d = DET.get(res["element_id"]) or {}
        print(f"  DET hints            : {[h['hint'] for h in d.get('hints', [])] or 'none'}")
        print(f"  DET facts            : style={(d.get('facts') or {}).get('assignment_style_of_element')}")
        if res["parse_status"] != "ok":
            print(f"  !! PARSE ERROR: {res['parse_detail']}\n  raw: {res['raw_response'][:300]}")
            continue
        fc = v.get("functionality_check", {})
        print(f"  VERIFIER functionality: {fc.get('status')} [{fc.get('evidence_type')}] "
              f"conf={fc.get('confidence')}")
        print(f"     rtl_evidence: {str(fc.get('rtl_evidence'))[:170]}")
        print(f"     rtl_location: {fc.get('rtl_location')}   trace={len(fc.get('rtl_trace') or [])} hops")
        if fc.get("interpretation"): print(f"     interpretation: {fc['interpretation'][:170]}")
        for c in v.get("relationship_checks", []):
            print(f"  VERIFIER rel {c.get('type')} -> {c.get('target')}: {c.get('status')} "
                  f"[{c.get('evidence_type')}] @ {c.get('rtl_location')}")
            print(f"     {str(c.get('rtl_evidence'))[:170]}")
        ec = v.get("evidence_check", {})
        print(f"  VERIFIER evidence     : {ec.get('status')} [{ec.get('evidence_type')}]")
        print(f"  OVERALL               : {v.get('overall_status')}  errors={len(v.get('errors') or [])}")
        for e in (v.get("errors") or []):
            print(f"     - {e.get('error_type')} / {e.get('severity')} / {e.get('status')}: "
                  f"{str(e.get('explanation'))[:180]}")

if SMOKE_ENABLED:
    inspect(SMOKE, SMOKE_SET)
    print("=" * 100)
    print("token usage / cost for the smoke set:")
    print(json.dumps(cost(), indent=2))
    print()
    print("projected full run:")
    print(json.dumps(project_cost(len(PARSED), n_elements_done=len(SMOKE)), indent=2))

smoke set disabled (SMOKE_ENABLED = False). Last result: 8/8 parsed, 0 API errors, 0 escalations, max trace depth 3.
8 elements are selected and ready if you re-enable it.


**What to look for, in order of importance**

1. Does any status say `CONTRADICTED` where the RTL merely fails to show something? That is
   the failure mode the whole design guards against.
2. Did a review hint get echoed back as the justification? Justifications must cite RTL.
3. Is the non-spec relationship type treated as an error *because* it is non-spec?
4. Is `evidence_type` honest — are `INTERPRETATION` calls labelled as such, or dressed up as
   `DIRECT_RTL`?
5. Does `rtl_trace` actually contain the hops, and do the line numbers land on real statements?
6. On the ordinary address field, is the verifier inventing problems? Over-flagging the
   baseline case means the prompt is still pushing toward error generation.

If those all look right, scale up. If not, fix the prompt before spending the full run.

## 9. Aggregation and recurring error patterns

Section 21 of the spec: aggregate recurring parser errors so the **parser prompt** can be
improved. The output describes patterns; it never rewrites the prompt automatically.

Pattern detection is deterministic — it reads the verifier's own structured fields plus the
deterministic facts, so a pattern count is reproducible rather than a second LLM judgement.

In [17]:
def _norm_key(s):
    """Stable join key for a claim.

    `claim_key` is built from strings the MODEL wrote, and the model reformats
    them freely between runs: `ISOLATED -> ` vs `ISOLATED -> (none)`,
    `bus_req_i.addr(hi downto 2)` vs `bus_req_i.addr (hi downto 2)`. Joining two
    runs on the raw string turns pure formatting into apparent disagreement.
    Normalise case, whitespace, empty-target spellings, and target order."""
    s = re.sub(r"\s+", " ", str(s or "")).strip().lower()
    if "->" in s:
        lhs, rhs = s.split("->", 1)
        # The parser NEVER puts parentheses in a target (verified: 0 of 68 targets in
        # neorv32_imem -- bit ranges live in a separate "bits" field). So any "(...)"
        # on the target side is the VERIFIER's own annotation: a bit range like
        # "(bits 7 downto 0)", or a placeholder like "(no targets)". It is not part of
        # the claim. Two runs differing only by whether the model volunteered a bit
        # range are making the same claim, and must join.
        rhs = re.sub(r"\([^)]*\)|\[[^\]]*\]", " ", rhs)
        rhs = re.sub(r"\b(none|n/a|null|no targets?|empty)\b", " ", rhs)
        tgts = sorted(t.strip() for t in re.split(r"[,;]", rhs) if t.strip())
        s = re.sub(r"\s+", " ", lhs).strip() + " -> " + "|".join(tgts)
    else:
        s = re.sub(r"\b(none|n/a|null)\b", " ", s)
        s = re.sub(r"\s+", " ", s).strip()
    return s

PARSED_BY_ID = {r["element_id"]: r for r in PARSED}

def flatten(results):
    """One row per claim. Elements whose response failed to parse are emitted with
    status PARSE_ERROR rather than None -- a NaN status is dropped by the pivot in
    status_table(), which made failed runs look cleaner than they were."""
    rows = []
    for r in results:
        v = r.get("verification") or {}
        failed = r.get("parse_status") != "ok"
        base = {k: r[k] for k in ["element_id","module","entity","element","rtl_file",
                                  "parse_status","cached"]}
        base["run_tag"] = r.get("run_tag")
        base["coverage_complete"] = (r.get("claim_coverage") or {}).get("complete")
        def add(field, key, status, node=None):
            node = node or {}
            rows.append({**base, "field": field, "claim_key": key,
                         "claim_key_norm": _norm_key(key),
                         "status": status or ("PARSE_ERROR" if failed else None),
                         # V-1 measured confidence as the best available predictor of
                         # whether a claim survives a repeat: 6% flip rate at >=0.95
                         # against 32% below 0.90. Keeping it (and evidence_type) only in
                         # per_file/*.json meant the main analysis table could not filter
                         # on the one column that separates solid findings from noise.
                         "confidence": node.get("confidence"),
                         "evidence_type": node.get("evidence_type"),
                         "occ": None,
                         "overall": v.get("overall_status")
                                    or ("PARSE_ERROR" if failed else None)})
        add("functionality", "", (v.get("functionality_check") or {}).get("status"),
            v.get("functionality_check"))
        if failed:
            # A failed element used to contribute functionality+evidence rows but NO
            # role/relationship rows, because those were read off an empty verification
            # dict. The four fields then had different denominators and nothing said so:
            # corpus-wide the parser makes 2.27 roles and 2.21 relationship pairs per
            # element, so each failure quietly removed ~4.5 rows from two of the four
            # fields. Emit one PARSE_ERROR row per PARSER claim instead.
            rec = PARSED_BY_ID.get(r["element_id"]) or {}
            for role in (rec.get("role") or []):
                add("role", role, None)
            for rel in (rec.get("relationship") or []):
                tg = rel.get("targets") or [None]
                for x in tg:
                    add("relationship", f"{rel.get('type')} -> {x if x else ''}", None)
        else:
            for c in v.get("role_checks", []) or []:
                add("role", c.get("role"), c.get("status"), c)
            for c in v.get("relationship_checks", []) or []:
                add("relationship", f"{c.get('type')} -> {c.get('target')}", c.get("status"), c)
        add("evidence", "", (v.get("evidence_check") or {}).get("status"),
            v.get("evidence_check"))
    df = pd.DataFrame(rows)
    if len(df):
        # 11 elements in this corpus legitimately repeat a (type, target) pair. Without
        # an occurrence index a run-to-run join cross-products those copies and invents
        # status transitions neither run produced.
        df["occ"] = df.groupby(["element_id", "field", "claim_key_norm"]).cumcount()
    return df

def error_summary(results):
    """Instance counts, WITH the denominators. Bare four-digit integers over 18 modules
    are unreadable: neorv32_bus supplies ~half of every one of them."""
    c = Counter(); sev = Counter()
    for r in results:
        for e in ((r.get("verification") or {}).get("errors") or []):
            c[e.get("error_type")] += 1; sev[(e.get("error_type"), e.get("severity"))] += 1
    n = max(1, len(results))
    top = Counter(r["module"] for r in results).most_common(1)
    return {"n_elements": len(results),
            "largest_module": {"module": top[0][0], "elements": top[0][1],
                               "share": round(top[0][1] / n, 3)} if top else None,
            "error_summary": dict(c.most_common()),
            "errors_per_100_elements": {k: round(100.0 * v / n, 1) for k, v in c.most_common()},
            "by_severity": {f"{k[0]}|{k[1]}": v for k, v in sev.most_common()}}

# One entry per error category the verifier is allowed to emit. Names match that
# vocabulary EXACTLY. The previous version used interpretive names
# ("omitted_conditional_branch") that asserted a sub-cause the error type does not
# guarantee, and carried a duplicate key that silently overwrote its twin.
#
# Counting unit: pattern_report counts ELEMENTS exhibiting a category at least
# once; error_summary counts ERROR INSTANCES. Different numbers, on purpose.
# Neither is a second LLM judgement -- both read the verifier's own structured
# `errors` array, so both are reproducible from a saved run.
def _has(et):
    return lambda r: any(e.get("error_type") == et
                         for e in ((r.get("verification") or {}).get("errors") or []))

PATTERNS = {et: _has(et) for et in sorted(ERROR_TYPES)}

# Severity/status refinements. Deliberately NOT keyword matches over the model's
# `explanation` prose -- a text detector is too brittle to be a research metric.
PATTERNS["_any_CONTRADICTED"] = lambda r: any(
    e.get("status") == "CONTRADICTED"
    for e in ((r.get("verification") or {}).get("errors") or []))
PATTERNS["_any_CRITICAL"] = lambda r: any(
    e.get("severity") == "CRITICAL"
    for e in ((r.get("verification") or {}).get("errors") or []))


def pattern_report(results):
    """Elements exhibiting each error category, most common first.

    `count` counts ELEMENTS, not errors -- an element with three
    CONDITIONAL_BEHAVIOR_ERRORs contributes 1. Use error_summary() for instance
    counts. `pct_of_elements` carries the denominator so the number cannot be
    quoted without it.
    """
    hits = Counter()
    examples = defaultdict(list)
    n = len(results)

    # Sort before sampling. Results arrive in thread-completion order, so the `examples`
    # column was drawn from whichever elements happened to finish first across 4 workers
    # -- non-reproducible between runs of the SAME data, and biased toward one module.
    results = sorted(results, key=lambda r: (r["module"], r["entity"], r["element"]))
    for r in results:
        for name, fn in PATTERNS.items():
            try:
                matched = bool(fn(r))
            except Exception:
                matched = False
            if matched:
                hits[name] += 1
                if len(examples[name]) < 5:
                    examples[name].append(f"{r['module']}/{r['entity']}/{r['element']}")

    return pd.DataFrame([
        {"pattern": name, "count": count,
         "pct_of_elements": round(100.0 * count / max(1, n), 1),
         "unit": "elements",
         "examples": ", ".join(examples[name])}
        for name, count in hits.most_common()
    ])

def status_table_by(flat, by="module"):
    """Same table, broken down. A corpus-level percentage is dominated by one entity:
    neorv32_bus is 51.5% of elements and neorv32_bus_io_switch alone is 34.5%, whose
    582 elements are a few structural shapes replicated 32 times. A single parser habit
    on those replicated ports can move any corpus number by ~30 points and read as a
    corpus-wide defect. Always look at this next to the blended table."""
    t = flat.pivot_table(index=[by, "field"], columns="status", values="element_id",
                         aggfunc="count", fill_value=0)
    t["total"] = t.sum(axis=1)
    for s in ["SUPPORTED", "PARTIALLY_SUPPORTED", "CONTRADICTED",
              "NOT_ESTABLISHED", "UNVERIFIABLE", "PARSE_ERROR"]:
        if s in t.columns:
            t[s + "_pct"] = (100 * t[s] / t["total"]).round(1)
    return t

def error_summary_by_module(results):
    """error_summary() with the denominators it is counting against."""
    out = {}
    per_mod = defaultdict(lambda: {"elements": 0, "errors": Counter()})
    for r in results:
        d = per_mod[r["module"]]
        d["elements"] += 1
        for e in ((r.get("verification") or {}).get("errors") or []):
            d["errors"][e.get("error_type")] += 1
    for mod, d in sorted(per_mod.items()):
        n = max(1, d["elements"])
        out[mod] = {"n_elements": d["elements"],
                    "errors": dict(d["errors"].most_common()),
                    "errors_per_100_elements": {k: round(100.0 * v / n, 1)
                                                for k, v in d["errors"].most_common()}}
    return out

def status_table(flat):
    t = flat.pivot_table(index="field", columns="status", values="element_id",
                         aggfunc="count", fill_value=0)
    t["total"] = t.sum(axis=1)
    for s in ["SUPPORTED","PARTIALLY_SUPPORTED","CONTRADICTED","NOT_ESTABLISHED","UNVERIFIABLE"]:
        if s in t.columns: t[s+"_pct"] = (100*t[s]/t["total"]).round(1)
    return t
print("aggregation ready")

aggregation ready


## 10. Save results and manifest (never overwrite; never touch data/)

In [18]:
def save_run(results, cfg=CONFIG, note=""):
    # HARD GUARANTEE: nothing is written under data/
    assert Path(cfg["parsed_dir"]).resolve() not in RUN_DIR.resolve().parents, "would write into data/"
    per = RUN_DIR / "per_file"; per.mkdir(exist_ok=True)
    by = defaultdict(list)
    # dict, not a linear scan per result: 1689 results x 1689 PARSED was ~2.9M compares.
    _SRC = {x["element_id"]: x["source_file"] for x in PARSED}
    for r in results:
        by[_SRC.get(r["element_id"], "unknown.json")].append(r)
    for src, rows in by.items():
        (per / f"{Path(src).stem}_verification.json").write_text(
            json.dumps(rows, indent=2, ensure_ascii=False), encoding="utf-8")

    flat = flatten(results)
    flat.to_csv(RUN_DIR / "claim_status.csv", index=False)
    status_table(flat).to_csv(RUN_DIR / "status_by_field.csv")
    # Per-module views. The blended corpus table is dominated by one entity, so these
    # are what make a single expensive pass yield per-module insight.
    status_table_by(flat, "module").to_csv(RUN_DIR / "status_by_module.csv")
    status_table_by(flat, "entity").to_csv(RUN_DIR / "status_by_entity.csv")
    (RUN_DIR / "error_summary_by_module.json").write_text(
        json.dumps(error_summary_by_module(results), indent=2), encoding="utf-8")
    pattern_report(results).to_csv(RUN_DIR / "error_patterns.csv", index=False)
    (RUN_DIR / "error_summary.json").write_text(
        json.dumps(error_summary(results), indent=2), encoding="utf-8")
    (RUN_DIR / "verifier_prompt.md").write_text(VERIFIER_SYSTEM, encoding="utf-8")

    try:
        git = subprocess.run(["git","rev-parse","HEAD"], capture_output=True, text=True,
                             cwd=str(HERE)).stdout.strip() or None
    except Exception: git = None
    manifest = {
        "run_dir": RUN_DIR.name,
        "timestamp_utc": datetime.now(timezone.utc).isoformat(),
        "note": note, "git_commit": git, "python": sys.version.split()[0],
        "run_tag": cfg.get("run_tag"),
        "parser_prompt_version": cfg["parser_prompt_version"],
        "parsed_data_version": cfg["parsed_data_version"],
        "rtl_dataset_version": cfg["rtl_dataset_version"],
        "verifier_prompt_version": VERIFIER_PROMPT_VERSION,
        # Retrieval settings are part of the arm, not incidental tuning: they decide
        # whether the verifier could treat absence as evidence.
        "retrieval": {
            "prefer_full_architecture": cfg.get("prefer_full_architecture"),
            "ctx_max_chars": cfg.get("ctx_max_chars"),
            "ctx_window_lines": cfg.get("ctx_window_lines"),
            "ctx_max_hits": cfg.get("ctx_max_hits"),
            "context_modes": dict(Counter(r.get("context_mode") for r in results)),
        },
        "verifier_prompt_sha256": VERIFIER_PROMPT_SHA,
        "llm": {"provider": cfg["provider"], "model": cfg["model"],
                "reasoning": USE_REASONING,
                "reasoning_effort": cfg["reasoning_effort"] if USE_REASONING else None,
                "max_output_tokens": cfg["max_output_tokens"],
                "max_output_tokens_ceiling": cfg["max_output_tokens_ceiling"],
                "note": "Responses API; reasoning models accept no temperature or seed, "
                        "so run-to-run determinism comes from the response cache"},
        # This run only. cost(cfg) with no usage argument reads the session-wide
        # USAGE list, which double-counts as soon as the STAGE cell is run twice
        # in one kernel. LAST_RUN_USAGE is the slice run_verification just billed.
        "usage_and_cost": cost(cfg, usage=LAST_RUN_USAGE),
        "usage_and_cost_session_cumulative": cost(cfg),
        "escalation_waste": escalation_waste(cfg, usage=LAST_RUN_USAGE),
        "prices_per_1m": {"in": cfg["price_in"], "cached_in": cfg["price_cached_in"],
                          "out": cfg["price_out"]},
        "counts": {"elements_verified": len(results),
                   "parse_ok": int(sum(1 for r in results if r["parse_status"]=="ok")),
                   "budget_escalations": int(sum(r.get("escalations",0) for r in results)),
                   "still_incomplete": int(sum(1 for r in results if r.get("incomplete"))),
                   # parse_ok alone cannot separate these: a malformed answer, an API
                   # failure and a crashed worker all just fail to be "ok".
                   "api_errors": int(sum(1 for r in results if r.get("api_error"))),
                   "harness_errors": int(sum(1 for r in results
                                             if r.get("parse_status") == "harness_error")),
                   "parse_errors": int(sum(1 for r in results
                                           if r.get("parse_status") == "parse_error")),
                   "unverifiable_mapping": int(sum(
                       1 for r in results if (r.get("verification") or {}).get("overall_status")=="UNVERIFIABLE")),
                   "claim_coverage_incomplete": int(sum(
                       1 for r in results
                       if not (r.get("claim_coverage") or {}).get("complete", True))),
                   "rel_pairs_lost_to_merging": int(sum(
                       max(0, (r.get("claim_coverage") or {}).get("rel_pairs_expected", 0)
                              - (r.get("claim_coverage") or {}).get("rel_checks_returned", 0))
                       for r in results))},
        "inputs": [{"json": s, "rtl": sorted({r["rtl_file"] for r in rows if r["rtl_file"]}),
                    "n_elements": len(rows)} for s, rows in sorted(by.items())],
        "rtl_sha256": {k: v["sha256"][:16] for k, v in RTL_FILES.items()},
        "anti_drift": "no prior verification result was read as evidence; guard asserted per payload",
    }
    (RUN_DIR / "manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
    print("saved ->", RUN_DIR)
    return manifest

# MANIFEST = save_run(RESULTS, note="first full verification pass")
print("save_run() ready")

save_run() ready


## 11. Full run

Running the notebook top-to-bottom stops after the **8-call smoke set**. The full pass is
deliberately left commented so it cannot fire by accident.

Work up in stages — each stage is cached, so nothing is ever paid for twice:

In [19]:
# =============================================================================
#  STAGE = 0  -> no API calls. Prints the plan and per-module counts only.
#  STAGE = 1  -> one module (set MOD). ~27 calls for neorv32_imem, ~$0.20.
#  STAGE = 2  -> full pass, 1689 elements. Resumable; cached calls are free.
#
#    COST : at the measured VP5 rate (run_010: 27 elements, $0.5011, 0 escalations)
#           that is $0.0186/element -> ~$31. Extrapolated from ONE 27-element module,
#           so an order of magnitude, not a quote.
#    TIME : run_010 averaged 65s per call (max 119s) = 31.9 compute-hours for 1689
#           elements. At parallel_workers=8 that is ~4 HOURS of wall clock.
#    IF IT DIES : nothing is re-paid for. Every completed element is already in the
#           response cache, so re-running this cell replays them in seconds and calls
#           only what is missing. You lose the RESULTS variable, not the work.
# =============================================================================
STAGE = 2
MOD   = "neorv32_imem"

if STAGE == 0:
    print("STAGE = 0 : no API calls will be made. Set STAGE = 1 or 2 to run.")
elif STAGE in (1, 2):
    todo = [r for r in PARSED if r["module"] == MOD] if STAGE == 1 else PARSED
    label = f"module {MOD}" if STAGE == 1 else "full pass"
    print(f"STAGE {STAGE}: {label} -> {len(todo)} elements "
          f"(cached ones cost nothing)\n")
    RESULTS = run_verification(todo)
    flat = flatten(RESULTS)
    display(status_table(flat))
    if STAGE == 2:
        # The corpus-blended table is dominated by one entity: neorv32_bus is 51.5%
        # of elements and neorv32_bus_io_switch alone is 34.5%, and its 582 elements
        # are a few shapes replicated 32x. Read the per-module split next to it or a
        # single parser habit on dev_NN_* ports reads as a corpus-wide defect.
        print("\nPER MODULE — the corpus table above is mostly neorv32_bus:")
        display(status_table_by(flat, "module"))
    display(pattern_report(RESULTS))
    print(json.dumps(error_summary(RESULTS), indent=2))
    MANIFEST = save_run(RESULTS, note=f"stage {STAGE}: {label}, parser "
                                      f"{CONFIG['parser_prompt_version']}, verifier "
                                      f"{VERIFIER_PROMPT_VERSION}")
    # bill THIS run, not everything the kernel has done
    print(json.dumps(cost(CONFIG, usage=LAST_RUN_USAGE), indent=2))
    if STAGE == 1:
        print(json.dumps(project_cost(len(PARSED),
                                      n_elements_done=len(RESULTS)), indent=2))
else:
    raise ValueError("STAGE must be 0, 1 or 2")

print()
print(f"corpus: {len(PARSED)} elements across {len({r['module'] for r in PARSED})} modules")
for m, n in Counter(r["module"] for r in PARSED).most_common():
    print(f"   {m:28s} {n:5d}")

STAGE 2: full pass -> 1689 elements (cached ones cost nothing)

  25/1689
  50/1689
  75/1689
  100/1689
  125/1689
  150/1689
  175/1689
  200/1689
  225/1689
  250/1689
  275/1689
  300/1689
  325/1689
  350/1689
  375/1689
  400/1689
  425/1689
  450/1689
  475/1689
  500/1689
  525/1689
  550/1689
  575/1689
  600/1689
  625/1689
  650/1689
  675/1689
  700/1689
  725/1689
  750/1689
  775/1689
  800/1689
  825/1689
  850/1689
  875/1689
  900/1689
  925/1689
  950/1689
  975/1689
  1000/1689
  1025/1689
  1050/1689
  1075/1689
  1100/1689
  1125/1689
  1150/1689
  1175/1689
  1200/1689
  1225/1689
  1250/1689
  1275/1689
  1300/1689
  1325/1689
  1350/1689
  1375/1689
  1400/1689
  1425/1689
  1450/1689
  1475/1689
  1500/1689
  1525/1689
  1550/1689
  1575/1689
  1600/1689
  1625/1689
  1650/1689
  1675/1689
verified 1689 | parsed_ok 99.9% | cached 27 | budget escalations 0 | still incomplete 0
  !! CLAIM COVERAGE: 2/1689 element(s) returned fewer checks than the parser made clai

status,CONTRADICTED,NOT_ESTABLISHED,PARSE_ERROR,PARTIALLY_SUPPORTED,SUPPORTED,total,SUPPORTED_pct,PARTIALLY_SUPPORTED_pct,CONTRADICTED_pct,NOT_ESTABLISHED_pct
field,,,,,,,,,,
evidence,1,0,2,79,1607,1689,95.1,4.7,0.1,0.0
functionality,1,24,2,540,1122,1689,66.4,32.0,0.1,1.4
relationship,187,60,9,851,2627,3734,70.4,22.8,5.0,1.6
role,6,52,6,552,3219,3835,83.9,14.4,0.2,1.4



PER MODULE — the corpus table above is mostly neorv32_bus:


status                          CONTRADICTED  NOT_ESTABLISHED  PARSE_ERROR  \
module           field                                                       
neorv32_boot_rom evidence                  0                0            0   
                 functionality             0                0            0   
                 relationship              0                0            0   
                 role                      0                0            0   
neorv32_bus      evidence                  0                0            0   
...                                      ...              ...          ...   
neorv32_uart     role                      0                0            0   
neorv32_wdt      evidence                  0                0            0   
                 functionality             0                0            0   
                 relationship             11                2            0   
                 role                      0                0            0   

status                          PARTIALLY_SUPPORTED  SUPPORTED  total  \
module           field                                                  
neorv32_boot_rom evidence                         2         19     21   
                 functionality                    2         19     21   
                 relationship                     2         29     31   
                 role                             2         34     36   
neorv32_bus      evidence                        34        836    870   
...                                             ...        ...    ...   
neorv32_uart     role                            11        146    157   
neorv32_wdt      evidence                         0         39     39   
                 functionality                    4         35     39   
                 relationship                    34         83    130   
                 role                             3         75     78   

status                          SUPPORTED_pct  PARTIALLY_SUPPORTED_pct  \
module           field                                                   
neorv32_boot_rom evidence                90.5                      9.5   
                 functionality           90.5                      9.5   
                 relationship            93.5                      6.5   
                 role                    94.4                      5.6   
neorv32_bus      evidence                96.1                      3.9   
...                                       ...                      ...   
neorv32_uart     role                    93.0                      7.0   
neorv32_wdt      evidence               100.0                      0.0   
                 functionality           89.7                     10.3   
                 relationship            63.8                     26.2   
                 role                    96.2                      3.8   

status                          CONTRADICTED_pct  NOT_ESTABLISHED_pct  \
module           field                                                  
neorv32_boot_rom evidence                    0.0                  0.0   
                 functionality               0.0                  0.0   
                 relationship                0.0                  0.0   
                 role                        0.0                  0.0   
neorv32_bus      evidence                    0.0                  0.0   
...                                          ...                  ...   
neorv32_uart     role                        0.0                  0.0   
neorv32_wdt      evidence                    0.0                  0.0   
                 functionality               0.0                  0.0   
                 relationship                8.5                  1.5   
                 role                        0.0                  0.0   

status                          PARSE_ERROR_pct  
module           field                           
neorv32_boot_rom evide

,pattern,count,pct_of_elements,unit,examples
0,CONDITIONAL_BEHAVIOR_ERROR,413,24.5,elements,"neorv32_boot_rom/neorv32_boot_rom/rdata, neorv..."
1,RELATIONSHIP_ERROR,263,15.6,elements,"neorv32_boot_rom/neorv32_boot_rom/clk_i, neorv..."
2,ROLE_ERROR,181,10.7,elements,"neorv32_boot_rom/neorv32_boot_rom/clk_i, neorv..."
3,_any_CONTRADICTED,98,5.8,elements,"neorv32_boot_rom/neorv32_boot_rom/rstn_i, neor..."
4,FUNCTIONALITY_ERROR,88,5.2,elements,"neorv32_bus/neorv32_bus_gateway/b_req_o, neorv..."
5,_any_CRITICAL,79,4.7,elements,"neorv32_boot_rom/neorv32_boot_rom/rstn_i, neor..."
6,RELATIONSHIP_TARGET_ERROR,73,4.3,elements,"neorv32_bus/neorv32_bus_gateway/a_req_o.amo, n..."
7,MISSING_BEHAVIOR,46,2.7,elements,"neorv32_bus/neorv32_bus_io_switch/clk_i, neorv..."
8,EVIDENCE_ERROR,39,2.3,elements,neorv32_boot_rom/neorv32_boot_rom/bus_req_i.da...
9,UNVERIFIABLE_CLAIM,20,1.2,elements,neorv32_bus/neorv32_bus_io_switch/dev_13_rsp_i...


{
  "n_elements": 1689,
  "largest_module": {
    "module": "neorv32_bus",
    "elements": 870,
    "share": 0.515
  },
  "error_summary": {
    "CONDITIONAL_BEHAVIOR_ERROR": 551,
    "RELATIONSHIP_ERROR": 492,
    "ROLE_ERROR": 200,
    "FUNCTIONALITY_ERROR": 88,
    "RELATIONSHIP_TARGET_ERROR": 78,
    "MISSING_BEHAVIOR": 47,
    "EVIDENCE_ERROR": 39,
    "RESET_BEHAVIOR_ERROR": 22,
    "UNVERIFIABLE_CLAIM": 20,
    "CROSS_ENTITY_ERROR": 13,
    "COMBINATIONAL_REGISTER_ERROR": 2
  },
  "errors_per_100_elements": {
    "CONDITIONAL_BEHAVIOR_ERROR": 32.6,
    "RELATIONSHIP_ERROR": 29.1,
    "ROLE_ERROR": 11.8,
    "FUNCTIONALITY_ERROR": 5.2,
    "RELATIONSHIP_TARGET_ERROR": 4.6,
    "MISSING_BEHAVIOR": 2.8,
    "EVIDENCE_ERROR": 2.3,
    "RESET_BEHAVIOR_ERROR": 1.3,
    "UNVERIFIABLE_CLAIM": 1.2,
    "CROSS_ENTITY_ERROR": 0.8,
    "COMBINATIONAL_REGISTER_ERROR": 0.1
  },
  "by_severity": {
    "CONDITIONAL_BEHAVIOR_ERROR|MAJOR": 514,
    "RELATIONSHIP_ERROR|MAJOR": 291,
    "ROLE_ERROR

## 12. Workflow

```python
RESULTS  = run_verification()                 # all 1689 elements (cached, resumable)
flat     = flatten(RESULTS)
status_table(flat)                            # SUPPORTED / PARTIAL / CONTRADICTED per field
error_summary(RESULTS)                        # counts by error category and severity
pattern_report(RESULTS)                       # recurring parser-error patterns
MANIFEST = save_run(RESULTS, note="...")
```

Cost: one call per element, ~1689 for a full pass. Start with
`run_verification(PARSED[:25])`. Responses are cached on
`sha256(model + decoding + verifier_prompt + payload)`, so an interrupted pass resumes free,
and only elements whose RTL context or claims changed re-issue calls.

### The loop

```
RTL -> Parser -> Parsed JSON -> RTL-grounded Verifier -> Verified Error Report
    -> Aggregate Error Patterns -> Improve Parser Prompt -> Parser again -> Verifier again
```

Bump `parser_prompt_version` and point `parsed_dir` at the new parser output. A fresh
`run_NNN/` is created every time; nothing is overwritten and `data/` is never written to.

### What this notebook will not do

It will not tell you whether anything is a security asset. Every status is of the form
*"the parser claims X; the RTL supports / partially supports / contradicts / does not
establish X."* Absence of evidence is `NOT_ESTABLISHED`, never `CONTRADICTED`.